# Bài SMS (smishing): làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ phần phân tích của bài SMS, bắt đầu từ dữ liệu công khai, trong một file duy nhất. Mọi hàm đều được viết ngay trong
notebook, nên đọc từ trên xuống là thấy hết cách mỗi con số được tính ra. Notebook chạy độc lập: không cần repo nào khác, chỉ cần mạng để tải
dữ liệu lần đầu.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `sms_walkthrough.ipynb`, thư mục `expected/` và `requirements.txt`).
2. Cài thư viện đúng phiên bản: `pip install -r requirements.txt` (Python 3.12).
3. Mở notebook và chạy hết (Run All).

Lần chạy đầu, notebook tự tải:
- corpus `trannguyenthaituan/vietnamese_sms_dataset` trên Hugging Face (giấy phép CC-BY-4.0), ghim đúng revision `a90e2bd7` mà bài dùng, khoảng 1 MB;
- mô hình `vinai/phobert-base`, ghim revision `01daacda`, khoảng 540 MB.

Mọi thứ tải về và mọi file trung gian nằm trong thư mục `sms_work/` cạnh notebook. Lần chạy đầu mất khoảng 15 phút trên CPU (riêng tính
embedding PhoBERT cho bốn biến thể của corpus là khoảng 8 phút); các lần sau dùng lại embedding đã tính, khoảng 6 phút.

Thư mục `expected/` chứa các kết quả mà bài báo dùng (file JSON và CSV). Notebook chỉ đọc chúng để so; nếu thư mục này không có cạnh
notebook, notebook tự tải nó từ repo công khai.

### Notebook kiểm gì

Mỗi bước có ba phần: giải thích bước đó làm gì và vì sao bài cần nó, code làm lại trên dữ liệu thật, và phần kiểm so kết quả vừa tính với kết
quả kỳ vọng. Cuối mỗi bước có dòng `[step] x/y checks match`. Bước 15 gom tất cả lại. Nhiều phép kiểm so nguyên một file kết quả, từng trường một,
nên "khớp" ở đây nghĩa là notebook tính lại được đúng từng con số trong file đó.

Về độ chính xác: các bước không dùng mạng nơ-ron (đếm, chia train/test, TF-IDF, hồi quy logistic, bootstrap) cho kết quả giống hệt trên mọi máy.
Các bước dùng MLP chỉ giống hệt khi cùng phiên bản thư viện (xem `requirements.txt`) và cùng chế độ luồng của BLAS (thư viện nhân ma trận mà numpy
dùng); trên máy khác, chúng có thể lệch ở chữ số thứ ba, và bảng kiểm sẽ liệt kê từng trường lệch.

### Các bước

| Bước | Làm gì | So với |
|---|---|---|
| 1 | đọc corpus thô | |
| 2 | làm sạch, tách URL, chia train/test theo hash nội dung | `sms_messages.csv`, `sms_urls.csv`, `sms_snapshot.json` |
| 3 | ghép nội dung tin vào bảng tin | |
| 4 | không đọc chữ thì tách lớp được tới đâu | `shallow_cues.json`, `floor_test_predictions.csv` |
| 5 | nhãn bám theo lô thu thập tới mức nào | `batch_audit.json` |
| 6 | baseline TF-IDF + LogReg | `text_linear.json` |
| 7 đến 9 | phép so sánh đã đăng ký trước: kênh URL, text, fusion | `fusion_results.json`, `fusion_test_predictions.csv` |
| 10 | vì sao URL không giúp gì | `why_fusion.json` |
| 11 | luật trích URL chặt hơn | `fusion_results_psl.json` |
| 12 | chia lại để tin gần giống không nằm hai phía | `split_robustness.json` |
| 13 | kiểm tra theo lô thời gian; URL dạng ký tự, cắt cụt token, phân tích lỗi | `temporal_audit.json`, `extra_diagnostics.json` |
| 14 | ba thí nghiệm "pre-decision" | `diacritics_ablation.json`, `late_fusion.json`, `phobert_ft.json` |
| 15 | tổng hợp | |

### Thuật ngữ

| Từ | Nghĩa |
|---|---|
| ham / phishing | tin bình thường / tin lừa đảo (nhãn 0 / 1) |
| kênh (arm) | một cách biểu diễn tin để đưa vào mô hình: chỉ URL, chỉ chữ (text), hoặc cả hai (fusion) |
| F1 | điểm cân bằng giữa precision (báo phishing thì đúng bao nhiêu) và recall (bắt được bao nhiêu phishing); luôn tính cho lớp phishing |
| FPR | tỷ lệ tin ham bị báo nhầm là phishing |
| seed | số khởi tạo ngẫu nhiên; đổi seed thì MLP học ra hơi khác dù dữ liệu y nguyên |
| sàn (floor) | điểm mà một mô hình đạt được khi không đọc chữ nào; mô hình chữ phải vượt mức này mới có nghĩa |
| CI 95% | khoảng tin cậy 95%; nếu khoảng chứa 0 thì không nói được hai kênh khác nhau |
| post-hoc | phân tích làm sau khi đã thấy kết quả, không có trong bản đăng ký trước (`PREREG_smishing.md`) |
| BLAS | thư viện nhân ma trận mà numpy dùng; số luồng của nó ảnh hưởng tới chữ số cuối của MLP |

## Bước 0: Thiết lập

Ô đầu tìm thư mục gốc của repo và khai báo đường dẫn. Ô thứ hai định nghĩa các hàm dùng để kiểm:
- `check(step, name, mine, saved, tol)` ghi lại một phép so. Với `tol = 0`, hai giá trị phải bằng nhau tuyệt đối. Số đã được làm tròn 4 chữ số
  trong file kết quả thì notebook cũng làm tròn 4 chữ số trước khi so. Số không làm tròn (như kết quả bootstrap) được so với sai số 1e-9.
- `report(step)` in số phép kiểm khớp của một bước, kèm chi tiết từng chỗ lệch nếu có.
- `check_json(step, label, mine, saved_file)` so mọi trường của một kết quả notebook tính ra với một file JSON đã lưu.
- `one_thread()` giới hạn BLAS ở một luồng trong một khối code. MLP chỉ cho kết quả giống hệt nhau khi chạy cùng máy, cùng phiên bản thư
  viện và cùng số luồng. Phần lớn kết quả đã lưu được tính ở chế độ nhiều luồng mặc định, còn bốn kết quả (Bước 5, Bước 13 phần chẩn đoán
  phụ, Bước 14a và 14b) được tính với một luồng, nên notebook dùng đúng chế độ đó cho từng bước. Đừng đặt `OPENBLAS_NUM_THREADS=1` cho cả
  kernel: kênh fusion ở Bước 12 sẽ ra 0,9409 thay vì 0,942.

In [ ]:
import os, csv, json, re, math, hashlib, collections, statistics, unicodedata, tempfile, filecmp, warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

import urllib.request
from huggingface_hub import hf_hub_download

HF_DATASET, HF_DATASET_REV = "trannguyenthaituan/vietnamese_sms_dataset", "a90e2bd7e8df2376939658075ba8094dcee488ad"
PHOBERT_REPO, PHOBERT_REV = "vinai/phobert-base", "01daacda68afe13d83023d16ec647239e344a1e6"
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/sms/expected"
EXPECTED_FILES = ["sms_messages.csv", "sms_urls.csv", "sms_snapshot.json", "shallow_cues.json",
                  "floor_test_predictions.csv", "batch_audit.json", "text_linear.json", "fusion_results.json",
                  "fusion_test_predictions.csv", "why_fusion.json", "fusion_results_psl.json", "split_robustness.json",
                  "temporal_audit.json", "temporal_predictions.csv", "extra_diagnostics.json",
                  "diacritics_ablation.json", "late_fusion.json", "phobert_ft.json"]

WORK = Path.cwd() / "sms_work"          # downloads, embeddings and every file this notebook writes
WORK.mkdir(exist_ok=True)
RAW_DIR = WORK / "corpus"
for f in ("full_dataset.csv", "train.csv", "test.csv"):
    hf_hub_download(HF_DATASET, f, repo_type="dataset", revision=HF_DATASET_REV, local_dir=RAW_DIR)
SRC = RAW_DIR / "full_dataset.csv"

# The expected results: next to the notebook, or fetched from the public repository
EXPECTED = Path.cwd() / "expected"
if not all((EXPECTED / f).exists() for f in EXPECTED_FILES):
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    for f in EXPECTED_FILES:
        if not (EXPECTED / f).exists():
            urllib.request.urlretrieve(f"{EXPECTED_URL}/{f}", EXPECTED / f)
SAVED = SECTIONS = EXPECTED
TMP = WORK / "rebuilt"                  # files rebuilt here and compared byte for byte
TMP.mkdir(exist_ok=True)
saved = lambda name: json.load(open(SAVED / name, encoding="utf-8"))

print("corpus   :", RAW_DIR)
print("expected :", EXPECTED)
print("OPENBLAS_NUM_THREADS in this kernel:", os.environ.get("OPENBLAS_NUM_THREADS", "(unset, as intended)"))

In [ ]:
CHECKS = []

def _same(a, b):
    return a == b or (isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b))

def check(step, name, mine, ref, tol=0.0):
    # Record one comparison between a value computed here and the saved value
    if tol and isinstance(mine, (tuple, list)):
        ok = len(mine) == len(ref) and all(abs(a - b) <= tol for a, b in zip(mine, ref))
        mine, ref = tuple(round(v, 6) for v in mine), tuple(round(v, 6) for v in ref)
    elif tol:
        ok = abs(float(mine) - float(ref)) <= tol
    else:
        ok = _same(mine, ref)
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    # Per-step tally, with every mismatch in full
    rows_ = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows_ if not c["match"]]
    print(f"[{step}] {len(rows_) - len(bad)}/{len(rows_)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def _plain(o):
    # JSON round trip turns tuples into lists and numpy scalars into Python numbers, as json.dump does
    return json.loads(json.dumps(o, default=lambda v: v.item() if hasattr(v, "item") else str(v)))

def flat(d, prefix=""):
    # {"a": {"b": [ {...} ]}} -> {"a.b[0]...": value}, so two results can be compared field by field
    if isinstance(d, dict):
        for k, v in d.items():
            yield from flat(v, f"{prefix}.{k}" if prefix else str(k))
    elif isinstance(d, list) and d and all(isinstance(x, (dict, list)) for x in d):
        for i, v in enumerate(d):
            yield from flat(v, f"{prefix}[{i}]")
    else:
        yield prefix, d

def check_json(step, label, mine, saved_file):
    # Every field of a result computed here against the saved JSON file
    new = dict(flat(_plain(mine)))
    ref = dict(flat(json.load(open(saved_file, encoding="utf-8"))))
    keys = sorted(set(new) | set(ref))
    diff = [k for k in keys if not _same(new.get(k, "<missing>"), ref.get(k, "<missing>"))]
    check(step, f"{label}: fields that differ (of {len(keys)})", len(diff), 0)
    for k in diff[:10]:
        print(f"    differs: {k}: saved={ref.get(k, '<missing>')!r} notebook={new.get(k, '<missing>')!r}")

def same_file(a, b):
    return filecmp.cmp(a, b, shallow=False)

def one_thread():
    # Limit BLAS to one thread inside a `with` block
    return threadpool_limits(limits=1, user_api="blas")

## Bước 1: Corpus thô

Dữ liệu là bộ `trannguyenthaituan/vietnamese_sms_dataset` trên Hugging Face (giấy phép CC-BY-4.0), file `full_dataset.csv`, đúng bản
revision `a90e2bd7` mà bài ghim. Mỗi dòng có `message_id`, `message`, `label` (0 là ham, 1 là phishing) và `date`.

Trước khi đọc tiếp, cần biết ba đặc điểm của corpus này:
- Nhà phát hành đã che (redact) thông tin cá nhân trước khi đăng. Số tiền, ngày, tên... được thay bằng token như `[MONEY]`, `[DATE]`,
  `[QC]` (quảng cáo), `[TB]` (thông báo). Bước 4 sẽ cho thấy chỉ riêng các token này đã tách hai lớp khá tốt.
- Cột `date` là ngày nhập lô dữ liệu, không phải ngày nhận tin. Cả corpus gần 3.000 tin chỉ có 21 ngày khác nhau, và ngày cuối cùng trùng
  ngày repo trên Hugging Face được tạo. Bài đọc cột này đúng nghĩa đó.
- Nhãn đi theo lô. Bảng dưới cho thấy có lô gần như toàn phishing và có lô toàn ham. Bước 5 đo xem điều này ảnh hưởng thế nào.

In [ ]:
raw = pd.read_csv(SRC, encoding="utf-8-sig", dtype=str)
print("shape:", raw.shape, "| columns:", raw.columns.tolist())
print(raw["label"].value_counts().rename({"0": "ham", "1": "phishing"}).to_string())
print("distinct values of `date`:", raw["date"].nunique())

# Batch (date) x class: the class mix is far from uniform across batches
by_batch = pd.crosstab(raw["date"], raw["label"]).rename(columns={"0": "ham", "1": "phishing"})
by_batch = by_batch.loc[sorted(by_batch.index, key=lambda d: pd.to_datetime(d, dayfirst=True))]
by_batch["phishing_share"] = (by_batch["phishing"] / by_batch.sum(axis=1)).round(3)
by_batch

In [ ]:
# One message of each class, shortened (the redaction tokens are visible)
for lab, name in (("0", "ham"), ("1", "phishing")):
    t = raw.loc[raw.label == lab, "message"].iloc[0]
    print(f"[{name}] {t[:220]}{'...' if len(t) > 220 else ''}\n")

## Bước 2: Làm sạch, tách URL, chia train/test

Bước này không huấn luyện mô hình nào và không sửa nhãn. Nó làm bốn việc, lần lượt dưới đây, rồi ghép lại để dựng bảng tin, bảng URL và bảng
số liệu mô tả corpus.

### 2a. Chia train/test theo hash của nội dung tin

Nhà phát hành có sẵn `train.csv` và `test.csv`, nhưng một số tin xuất hiện ở cả hai file. Mô hình được chấm trên đúng câu nó đã học thì
điểm test sẽ cao hơn thực tế. Bài tự chia lại như sau:

- `sha(text)` tính SHA-1 của nội dung tin (đã bỏ khoảng trắng hai đầu) và giữ 16 ký tự đầu.
- `split_of(sha)` băm tiếp chuỗi `"sms-split:" + sha` thành một số từ 0 đến 1. Số nhỏ hơn `HOLDOUT = 0.20` thì tin vào test, còn lại
  vào train.

Vì chia theo nội dung, mọi bản sao của cùng một tin luôn nằm cùng một phía. Kết quả chia cũng giống nhau trên mọi máy mà không cần seed
hay file split riêng.

In [ ]:
HOLDOUT = 0.20

def sha(t):
    return hashlib.sha1((t or "").strip().encode("utf-8")).hexdigest()[:16]

def split_of(text_sha):
    # Keyed on the message text, so duplicates land together; stable across machines
    h = int(hashlib.sha1(("sms-split:" + text_sha).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "test" if h < HOLDOUT else "train"

a = "Quy khach vui long truy cap vietcombank.vn-gll.top de xac thuc"
b = "Quy khach vui long truy cap vietcombank.vn-gll.top de xac thuc"   # exact copy
c = "Quy khach vui long truy cap vietcombank.vn-gll.top de xac thuc!"  # one character added
for t in (a, b, c):
    print(f"{sha(t)}  ->  {split_of(sha(t))}")

Hai bản sao y hệt nhau cho cùng hash nên nằm cùng phía. Nhưng chỉ cần thêm một ký tự là hash đổi hoàn toàn. Vậy cách chia này chặn được
tin trùng nguyên văn, còn tin gần giống (chỉ khác số tiền hay ngày) thì vẫn có thể nằm ở hai phía. Bước 4 đo mức độ của chuyện này, và
Bước 12 chia lại để chặn nó.

### 2b. Nhận biết tin có dấu

`has_vietnamese_diacritic` chuẩn hoá chuỗi về dạng NFD. Ở dạng này mỗi chữ có dấu được tách thành chữ gốc cộng các dấu riêng, ví dụ "ý"
thành "y" và dấu sắc. Hàm tìm 8 loại dấu (5 dấu thanh, dấu mũ, dấu trăng, dấu móc). Chữ `đ` được xét riêng, vì NFD không tách `đ` thành
`d` cộng một dấu.

In [ ]:
VI_MARKS = set("̛̣̀́̃̉̂̆")

def has_vietnamese_diacritic(text):
    nfd = unicodedata.normalize("NFD", text or "")
    return "đ" in (text or "").lower() or any(ch in VI_MARKS for ch in nfd)

print("marks:", [unicodedata.name(ch) for ch in sorted(VI_MARKS)])
for t in ["Quy khach", "Quý khách", "dang ky", "đang ky", "Đăng ký"]:
    print(f"{t!r:14} -> {has_vietnamese_diacritic(t)}")

### 2c. Tách URL, kể cả khi không có `http://`

Tin lừa đảo hay viết kiểu `truy cap vietcombank.vn-gll.top`, không có `http://`. Vì vậy regex `CAND` bắt cả tên miền đứng một mình. Với mỗi
chỗ khớp có ba việc:
- `registered_domain` lấy tên miền gốc (gọi là apex) theo Public Suffix List, tức danh sách các đuôi tên miền có thật như `.vn`, `.com.vn`,
  `.top`. Ví dụ apex của `vietcombank.vn-gll.top` là `vn-gll.top`. Danh sách này lấy từ bản đóng gói sẵn trong thư viện `tldextract`
  (không tải từ mạng), nên kết quả không đổi theo thời gian.
- `has_public_suffix` kiểm tra chỗ khớp có kết thúc bằng một đuôi thật không. Những mẩu như `tp.hcm` khớp regex nhưng không phải URL, nên
  được đánh `valid = 0`. Các dòng này vẫn được giữ, vì kênh URL trong bản đăng ký đã dùng chúng.
- `is_short` đánh dấu link rút gọn. Có hai danh sách: `SHORTENERS_REG` là danh sách trong bản đăng ký (lỡ tính cả `zalo.me`, vốn là link mở
  trang Zalo chứ không phải dịch vụ rút gọn), và `SHORTENERS` là danh sách đã sửa. Giữ cả hai để kết quả đăng ký vẫn làm lại được.

In [ ]:
import tldextract

_PSL = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)  # bundled snapshot, offline

def _apex(result):
    new = getattr(result, "top_domain_under_public_suffix", None)
    return result.registered_domain if new is None else new

def registered_domain(host):
    return _apex(_PSL(host)).lower() or host

def has_public_suffix(host):
    return bool(_PSL(host).suffix)

CAND = re.compile(r"(?:https?://)?((?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+[a-z]{2,})(?:/[^\s,;)\]]*)?", re.I)
SHORTENERS_REG = {"bit.ly", "tinyurl.com", "goo.gl", "t.co", "rb.gy", "shorturl.at", "cutt.ly",
                  "is.gd", "ow.ly", "s.id", "zalo.me", "page.link", "me.qr", "link.vn",
                  "shorturl.asia", "buff.ly", "rebrand.ly"}
SHORTENERS = (SHORTENERS_REG - {"zalo.me"}) | {"t.ly", "ln.run", "qrco.de", "onelink.me", "onelink.to", "go.link",
                                              "lzd.co", "grb.to", "lnkd.in", "cps.onl", "o2o.vn", "id.vin"}
OPERATOR_DEEP_LINKS = {"myvt.page.link", "viettelmoney.go.link", "myviettel.go.link", "vtmoney.onelink.me"}
PII_TOKEN = re.compile(r"\[[A-Z_]+\]")
RESIDUAL = {"phone_vn": re.compile(r"(?<![0-9])0[0-9]{8,10}(?![0-9])"),
            "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")}

def hosts_in(msg):
    # Every regex match, with a flag saying whether the host ends in a real public suffix
    out = []
    for m in CAND.finditer(msg):
        host = m.group(1).lower()
        apex = registered_domain(host)
        if apex and "." in apex:
            out.append((host, apex, has_public_suffix(host)))
    return out

def is_short(host, apex, names=SHORTENERS):
    return apex in names or host.endswith(".page.link")

msg = "Truy cap vietcombank.vn-gll.top hoac bit.ly/abc, ho tro zalo.me/123 tai tp.hcm"
print(pd.DataFrame([{"host": h, "apex": ap, "valid_url": ok, "short_corrected": is_short(h, ap),
                     "short_registered": is_short(h, ap, SHORTENERS_REG)} for h, ap, ok in hosts_in(msg)]).to_string(index=False))
print("\nremoved from the corrected list:", sorted(SHORTENERS_REG - SHORTENERS))
print("added to the corrected list    :", sorted(SHORTENERS - SHORTENERS_REG))

### 2d. Dựng bảng tin và bảng URL

Mỗi tin là một dòng của bảng tin. Bảng cố ý không lưu nội dung tin mà chỉ lưu `text_sha1`, để repo không đăng lại corpus từng dòng. Bước nào
cần nội dung thì nối lại với file gốc qua `message_id`.

| Cột | Nghĩa |
|---|---|
| `text_sha1`, `split` | `sha(message)` và `split_of(text_sha1)` |
| `chars`, `accented` | số ký tự; tin có dấu hay không |
| `pii_tokens` | các token che có trong tin, ví dụ `[DATE] [MONEY]` |
| `residual_pii` | số điện thoại hoặc email còn sót sau khi che, để kiểm việc che |
| `n_urls`, `n_shortened` | đếm theo luật trong bản đăng ký (tính cả mẩu không phải URL) |
| `n_urls_valid`, `n_shortened_valid` | đếm theo luật đã sửa (chỉ URL có đuôi thật) |
| `apexes` | các tên miền gốc hợp lệ trong tin |

Notebook ghi hai bảng ra thư mục tạm theo đúng định dạng của file đã lưu, rồi so từng byte.

In [ ]:
src_rows = list(csv.DictReader(open(SRC, newline="", encoding="utf-8-sig")))
msg_rows, url_rows = [], []
for r in src_rows:
    text, lab = r.get("message") or "", r.get("label", "")
    h = sha(text)
    hh = hosts_in(text)
    hv = [(host, apex) for host, apex, ok in hh if ok]
    msg_rows.append({
        "message_id": r.get("message_id", ""), "date": r.get("date", ""), "text_sha1": h, "label": lab,
        "split": split_of(h), "chars": len(text), "accented": int(has_vietnamese_diacritic(text)),
        "pii_tokens": " ".join(sorted(set(PII_TOKEN.findall(text)))),
        "residual_pii": " ".join(sorted(k for k, rx in RESIDUAL.items() if rx.search(text))),
        "n_urls": len(hh), "n_shortened": sum(1 for host, apex, _ in hh if is_short(host, apex, SHORTENERS_REG)),
        "n_urls_valid": len(hv), "n_shortened_valid": sum(1 for host, apex in hv if is_short(host, apex)),
        "apexes": " ".join(sorted({a_ for _, a_, ok in hh if ok}))})
    for host, apex, ok in hh:
        url_rows.append({"message_id": r.get("message_id", ""), "label": lab, "host": host, "apex": apex,
                         "tld": apex.rsplit(".", 1)[-1], "valid": int(ok), "shortener": int(ok and is_short(host, apex)),
                         "shortener_reg": int(is_short(host, apex, SHORTENERS_REG))})

for name, data in (("sms_messages.csv", msg_rows), ("sms_urls.csv", url_rows)):
    with open(TMP / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(data[0].keys()))
        w.writeheader()
        w.writerows(data)
    check("2 import", f"{name} byte-identical", same_file(TMP / name, SAVED / name), True)

msgs = pd.DataFrame(msg_rows)
print(msgs.groupby("label")[["chars", "accented", "n_urls_valid", "n_shortened_valid"]].mean()
      .rename(index={"0": "ham", "1": "phishing"}).round(3))

### 2e. Số liệu mô tả corpus và mức trùng của split nhà phát hành

Phần này đếm các con số mà bài dùng để mô tả corpus: số tin, số nội dung khác nhau, số tin có URL, số link rút gọn (có và không tính link
riêng của nhà mạng Viettel), các đuôi tên miền phổ biến... Nó cũng đo lại xem split của nhà phát hành bị trùng bao nhiêu, và kiểm rằng
cách chia của bài không để nội dung nào nằm ở cả hai phía.

In [ ]:
def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

tr_c = collections.Counter(sha(x["message"]) for x in read_csv_rows(RAW_DIR / "train.csv"))
te_c = collections.Counter(sha(x["message"]) for x in read_csv_rows(RAW_DIR / "test.csv"))
both = set(tr_c) & set(te_c)
pub_leak_texts, pub_test_rows, pub_leak_rows = len(both), sum(te_c.values()), sum(te_c[k] for k in both)
print(f"publisher split: {pub_leak_texts} texts on both sides, touching {pub_leak_rows} of {pub_test_rows} test rows")

by_split = collections.defaultdict(set)
for m in msg_rows:
    by_split[m["split"]].add(m["text_sha1"])
own_leak = len(by_split["train"] & by_split["test"])
print("texts on both sides of our split:", own_leak)

def per_label(pred):
    return {lab: sum(1 for m in msg_rows if m["label"] == lab and pred(m)) for lab in ("0", "1")}

n_lab = {lab: sum(1 for m in msg_rows if m["label"] == lab) for lab in ("0", "1")}
valid_by, reg_by = collections.defaultdict(list), collections.defaultdict(list)
for u in url_rows:
    reg_by[u["label"]].append(u)
    if u["valid"]:
        valid_by[u["label"]].append(u)

def short_hosts(v):
    return collections.Counter(u["host"] if u["host"] in OPERATOR_DEEP_LINKS else u["apex"]
                               for u in v if u["shortener"]).most_common()

snap = {
    "source": "trannguyenthaituan/vietnamese_sms_dataset full_dataset.csv (CC-BY-4.0)",
    "messages": len(msg_rows), "unique_texts": len({m["text_sha1"] for m in msg_rows}),
    "exact_duplicates": len(msg_rows) - len({m["text_sha1"] for m in msg_rows}),
    "ham": n_lab["0"], "phishing": n_lab["1"],
    "with_url": per_label(lambda m: m["n_urls_valid"] > 0),
    "urls": {k: len(v) for k, v in valid_by.items()},
    "unique_apexes": {k: len({u["apex"] for u in v}) for k, v in valid_by.items()},
    "shortened_urls": {k: sum(u["shortener"] for u in v) for k, v in valid_by.items()},
    "shortened_hosts": {k: short_hosts(v) for k, v in valid_by.items()},
    "operator_deep_links": sorted(OPERATOR_DEEP_LINKS),
    "shortened_urls_without_operator_link": {k: sum(u["shortener"] for u in v if u["host"] not in OPERATOR_DEEP_LINKS)
                                             for k, v in valid_by.items()},
    "urls_without_operator_link": {k: sum(1 for u in v if u["host"] not in OPERATOR_DEEP_LINKS) for k, v in valid_by.items()},
    "zalo_me_urls": {k: sum(1 for u in v if u["apex"] == "zalo.me") for k, v in valid_by.items()},
    "top_tlds": {k: collections.Counter(u["tld"] for u in v).most_common(6) for k, v in valid_by.items()},
    "registered_extractor": {
        "with_url": per_label(lambda m: m["n_urls"] > 0),
        "urls": {k: len(v) for k, v in reg_by.items()},
        "non_url_fragments": {k: sum(1 for u in v if not u["valid"]) for k, v in reg_by.items()},
        "fragment_hosts": collections.Counter(u["host"] for v in reg_by.values() for u in v if not u["valid"]).most_common(12),
        "messages_url_only_by_fragment": per_label(lambda m: m["n_urls"] > 0 and m["n_urls_valid"] == 0),
        "shortened_urls": {k: sum(u["shortener_reg"] for u in v) for k, v in reg_by.items()},
        "unique_apexes": {k: len({u["apex"] for u in v}) for k, v in reg_by.items()}},
    "residual_pii_messages": sum(1 for m in msg_rows if m["residual_pii"]),
    "pii_token_kinds": len({t for m in msg_rows for t in m["pii_tokens"].split()}),
    "own_split": {s: len(v) for s, v in sorted(by_split.items())},
    "own_split_leak_texts": own_leak,
    "published_split_leak_texts": pub_leak_texts, "published_split_leak_rows": pub_leak_rows,
    "published_test_rows": pub_test_rows}
print({k: snap[k] for k in ("messages", "unique_texts", "exact_duplicates", "ham", "phishing", "with_url", "urls")})
print("top TLDs:", snap["top_tlds"])
check_json("2 import", "sms_snapshot.json", snap, SAVED / "sms_snapshot.json")
report("2 import")

## Bước 3: Ghép lại nội dung tin

Các bước huấn luyện cần nội dung tin, nên notebook ghép bảng tin với file gốc qua `message_id`. Từ đây có năm biến dùng chung cho mọi bước:
`rows` (bảng tin, mỗi dòng là một dict), `texts`, `y`, `tr_mask`, `te_mask`. `groups` là nội dung (dạng hash) của từng dòng test, dùng cho
bootstrap ở Bước 9.

`rows` được đọc lại từ file CSV vừa ghi ở Bước 2, để mọi giá trị đều ở dạng chuỗi như khi đọc từ đĩa. Một vài phép tính phía sau (ví dụ
`int(r["chars"])`) dựa vào điều đó.

In [ ]:
MSG_FILE = TMP / "sms_messages.csv"
raw_text = {r["message_id"]: r["message"] for r in src_rows}
rows = list(csv.DictReader(open(MSG_FILE, newline="", encoding="utf-8")))
texts = [raw_text.get(r["message_id"], "") for r in rows]
y = np.array([int(r["label"]) for r in rows])
split = np.array([r["split"] for r in rows])
tr_mask, te_mask = split == "train", split == "test"
te_idx = np.where(te_mask)[0]
groups = [rows[i]["text_sha1"] for i in te_idx]
print(f"{len(texts)} messages | train {tr_mask.sum()} rows / test {te_mask.sum()} rows | "
      f"distinct test texts {len(set(groups))} | phishing in test {y[te_mask].sum()}")

## Bước 4: Không đọc chữ thì tách lớp được tới đâu?

Đây là phát hiện quan trọng nhất về corpus. Nếu token che (`[QC]`, `[TB]`, `[DATE]`...) xuất hiện nhiều ở lớp này mà ít ở lớp kia, mô hình
có thể phân loại chỉ bằng cách đếm token, không cần hiểu chữ nào.

Bước này làm sáu việc:
1. Đếm tỷ lệ xuất hiện của mỗi token, tỷ lệ tin có dấu và độ dài tin, theo từng lớp.
2. Huấn luyện LogReg trên những đặc điểm không phải ngôn ngữ: chỉ token, chỉ độ dài, dấu cộng độ dài, và cả ba (`all_shallow`). F1 của
   `all_shallow` là "sàn": mô hình đọc chữ phải vượt mức này thì mới có thể nói nó hiểu nội dung tin.
3. Thêm sáu đặc điểm hình thức (có URL thật không, có `http` không, có thẻ trong ngoặc vuông ở đầu tin không, tỷ lệ chữ in hoa, số chữ số,
   số dòng) và thử cả gradient boosting thay cho LogReg, vì LogReg chỉ cho một mức thấp của thứ mà các đặc điểm này tách được.
4. Chạy baseline TF-IDF, giữ và xoá token che. Nếu xoá token mà điểm tụt mạnh thì phần lớn tín hiệu nằm ở token.
5. Đo rò gần trùng: bao nhiêu tin test có một tin rất giống nó trong train, ở các ngưỡng độ giống (cosine) 0,95, 0,90 và 0,80.
6. Đếm họ mẫu tin: hai tin giống nhau từ 0,80 trở lên thì xếp cùng họ. Nếu 800 tin phishing thực chất chỉ là 80 mẫu viết lại, thì lượng
   thông tin thật chỉ tương đương 80 tin. Kèm theo là bảng từ đặc trưng của mỗi lớp (log-odds có prior Dirichlet, cách xếp hạng từ không bị
   từ hiếm hay từ quá phổ biến chi phối).

In [ ]:
labs = ["0", "1"]
tot = collections.Counter(r["label"] for r in rows)
acc_cnt = collections.Counter()
for r in rows:
    acc_cnt[r["label"]] += int(r["accented"] or 0)
accented = {l: round(100 * acc_cnt[l] / tot[l], 1) for l in labs}

tok = {l: collections.Counter() for l in labs}
any_tok = collections.Counter()
for r in rows:
    ts = set((r["pii_tokens"] or "").split())
    any_tok[r["label"]] += bool(ts)
    for t in ts:
        tok[r["label"]][t] += 1
names = sorted({t for l in labs for t in tok[l]}, key=lambda t: -sum(tok[l][t] for l in labs))
per_token = {t: {l: round(100 * tok[l][t] / tot[l], 1) for l in labs} for t in names}
lens = collections.defaultdict(list)
for r in rows:
    lens[r["label"]].append(int(r["chars"] or 0))

print("share of messages carrying each token (%):")
display(pd.DataFrame(per_token).T.rename(columns={"0": "ham", "1": "phishing"}))
print("accented (%):", accented, "| median chars:", {l: int(statistics.median(lens[l])) for l in labs})

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

TOKRX = re.compile(r"\[[A-Z_]+\]")
BRACKET = re.compile(r"\[[^\]\n]{1,40}\]")
LEAD_TAG = re.compile(r"^\s*\[[^\]\n]{1,40}\]")

def fit_f1(train, test, featf):
    # Logistic regression on whatever featf returns, scored on the phishing class
    Xtr = np.array([featf(r) for r in train], dtype=float); ytr = np.array([int(r["label"]) for r in train])
    Xte = np.array([featf(r) for r in test], dtype=float); yte = np.array([int(r["label"]) for r in test])
    p = LogisticRegression(max_iter=2000, random_state=0).fit(Xtr, ytr).predict(Xte)
    return {"f1": round(float(f1_score(yte, p)), 4), "precision": round(float(precision_score(yte, p, zero_division=0)), 4),
            "recall": round(float(recall_score(yte, p)), 4), "n_train": len(ytr), "n_test": len(yte)}

def gbm_predict(train, test, featf):
    Xtr = np.array([featf(r) for r in train], dtype=float); ytr = np.array([int(r["label"]) for r in train])
    Xte = np.array([featf(r) for r in test], dtype=float)
    return HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr).predict(Xte)

def fit_f1_gbm(train, test, featf):
    yte = np.array([int(r["label"]) for r in test])
    p = gbm_predict(train, test, featf)
    return {"f1": round(float(f1_score(yte, p)), 4), "precision": round(float(precision_score(yte, p, zero_division=0)), 4),
            "recall": round(float(recall_score(yte, p)), 4), "n_train": len(train), "n_test": len(yte),
            "model": "HistGradientBoosting"}

def format_feats(r):
    # Message format, read off the characters without reading a word
    t = r["text"]
    letters = [c for c in t if c.isalpha()]
    return [float(int(r.get("n_urls_valid") or 0) > 0), float("http" in t.lower()), float(bool(LEAD_TAG.match(t))),
            sum(c.isupper() for c in letters) / max(len(letters), 1), sum(c.isdigit() for c in t) / 10.0, t.count("\n") / 1.0]

train_r = [r for r in rows if r["split"] == "train"]
test_r = [r for r in rows if r["split"] == "test"]
f_tok = lambda r: [1.0 if t in set((r["pii_tokens"] or "").split()) else 0.0 for t in names]
f_len = lambda r: [int(r["chars"] or 0) / 100.0]
f_acc = lambda r: [float(int(r["accented"] or 0))]
f_all = lambda r: f_acc(r) + f_len(r) + f_tok(r)
base = {"tokens": fit_f1(train_r, test_r, f_tok),
        "tokens_and_length": fit_f1(train_r, test_r, lambda r: f_tok(r) + f_len(r)),
        "length": fit_f1(train_r, test_r, f_len),
        "accent_and_length": fit_f1(train_r, test_r, lambda r: f_acc(r) + f_len(r)),
        "all_shallow": fit_f1(train_r, test_r, f_all)}
display(pd.DataFrame(base).T[["f1", "precision", "recall"]])

Sàn `all_shallow` khoảng 0,71, cao hơn kênh URL (khoảng 0,61, Bước 8). Tức là kênh URL còn thua một mô hình không đọc chữ nào. Điểm của kênh
text vì thế phải được hiểu là "vượt sàn bao nhiêu", chứ không phải bằng chứng PhoBERT hiểu tiếng Việt.

Ô dưới làm nốt phần còn lại: các đặc điểm hình thức, baseline TF-IDF, rò gần trùng, họ mẫu tin và bảng từ đặc trưng.

In [ ]:
with_text = [dict(r, text=raw_text[r["message_id"]]) for r in rows]
tr_t = [r for r in with_text if r["split"] == "train"]
te_t = [r for r in with_text if r["split"] == "test"]

def tfidf_baselines(train, test):
    # Word and char n-gram TF-IDF, with and without the redaction tokens
    ytr = [int(r["label"]) for r in train]; yte = [int(r["label"]) for r in test]
    out = {}
    grids = (("word", dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True), False),
             ("char", dict(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True), False),
             ("word_no_tokens", dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True), TOKRX),
             ("word_no_brackets", dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True), BRACKET))
    for name, kw, strip in grids:
        prep = (lambda t, rx_=strip: rx_.sub(" ", t)) if strip else (lambda t: t)
        v = TfidfVectorizer(**kw)
        A = v.fit_transform([prep(r["text"]) for r in train]); B = v.transform([prep(r["text"]) for r in test])
        out[name] = round(float(f1_score(yte, LogisticRegression(max_iter=3000, C=4, random_state=0).fit(A, ytr).predict(B))), 4)
    return out

def duplicate_leak(train, test):
    # Test messages whose closest training message reaches each cosine threshold
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
    A = v.fit_transform([r["text"] for r in train]); B = v.transform([r["text"] for r in test])
    mx = cosine_similarity(B, A).max(axis=1)
    out = {}
    for thr in (0.95, 0.90, 0.80):
        out[f"{thr:.2f}"] = {}
        for lab in labs:
            idx = [i for i, r in enumerate(test) if r["label"] == lab]
            n = int(sum(1 for i in idx if mx[i] >= thr))
            out[f"{thr:.2f}"][lab] = {"n": n, "of": len(idx), "pct": round(100 * n / max(len(idx), 1), 1)}
    return out

def families(msgs_):
    # Template families: connected components of the "80% similar" graph
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True).fit_transform(msgs_)
    n, lab = connected_components(csr_matrix(cosine_similarity(X) >= 0.80), directed=False)
    sizes = collections.Counter(lab.tolist())
    return {"messages": len(msgs_), "families": int(n), "pct": round(100 * n / max(len(msgs_), 1), 1),
            "singletons": int(sum(1 for v_ in sizes.values() if v_ == 1)), "largest": sorted(sizes.values(), reverse=True)[:5]}

def log_odds(rows_, top=12):
    # Log-odds ratio with an informative Dirichlet prior at 1% of the observed counts
    cv = CountVectorizer(analyzer="word", ngram_range=(1, 1), min_df=3, token_pattern=r"(?u)\b\w[\w\-\.]+\b")
    X = cv.fit_transform([r["text"].lower() for r in rows_])
    yy = np.array([int(r["label"]) for r in rows_])
    vocab = np.array(cv.get_feature_names_out())
    c1 = np.asarray(X[yy == 1].sum(0)).ravel().astype(float); c0 = np.asarray(X[yy == 0].sum(0)).ravel().astype(float)
    a_ = (c1 + c0) * 0.01
    n1, n0, A = c1.sum(), c0.sum(), a_.sum()
    d = np.log((c1 + a_) / (n1 + A - c1 - a_)) - np.log((c0 + a_) / (n0 + A - c0 - a_))
    z = d / np.sqrt(1.0 / (c1 + a_) + 1.0 / (c0 + a_))
    order = np.argsort(z)
    pick = lambda idx: [{"term": str(vocab[i]), "z": round(float(z[i]), 1), "phish": int(c1[i]), "ham": int(c0[i])} for i in idx]
    return {"phishing": pick(order[::-1][:top]), "ham": pick(order[:top])}

lead = {l: [r for r in with_text if r["label"] == l] for l in labs}
sender_tag = {l: round(100 * sum(1 for r in v if LEAD_TAG.match(r["text"]) and not TOKRX.match(r["text"].lstrip()))
                       / max(len(v), 1), 1) for l, v in lead.items()}
tb_rx = re.compile(r"(?<![\w\[])tb(?![\w\]])", re.I)
tb_usage = {l: {"prefix_msgs": sum(1 for r in v if "[TB]" in r["text"]),
                "bare_tb_msgs": sum(1 for r in v if tb_rx.search(r["text"]))} for l, v in lead.items()}
carrier_rx = re.compile(r"viettel|vinaphone|mobifone|vietnamobile|itelecom|myvt", re.I)
carrier = {l: round(100 * sum(1 for r in v if carrier_rx.search(r["text"])) / max(len(v), 1), 1) for l, v in lead.items()}

lang = {"tfidf": tfidf_baselines(tr_t, te_t),
        "nonlinear_floor": {"all_shallow_gbm": fit_f1_gbm(tr_t, te_t, f_all),
                            "all_shallow_format_lr": fit_f1(tr_t, te_t, lambda r: f_all(r) + format_feats(r)),
                            "all_shallow_format_gbm": fit_f1_gbm(tr_t, te_t, lambda r: f_all(r) + format_feats(r)),
                            "strict_format_gbm": fit_f1_gbm(tr_t, te_t, lambda r: f_acc(r) + f_len(r) + format_feats(r))},
        "sender_tag_pct": sender_tag, "tb_usage": tb_usage, "carrier_mention_pct": carrier,
        "near_duplicate_leak": duplicate_leak(tr_t, te_t),
        "families": {l: families([r["text"] for r in with_text if r["label"] == l]) for l in labs},
        "log_odds": log_odds(with_text), "n_joined": len(with_text)}

print("TF-IDF F1:", lang["tfidf"])
print("non-linear floors:", {k: v["f1"] for k, v in lang["nonlinear_floor"].items()})
print("\nnear-duplicate leak:")
display(pd.DataFrame({thr: {("ham" if l == "0" else "phishing"): f"{v['n']}/{v['of']} ({v['pct']}%)" for l, v in d.items()}
                      for thr, d in lang["near_duplicate_leak"].items()}))
print("template families:", {("ham" if k == "0" else "phishing"): f"{v['families']} families / {v['messages']} msgs, largest {v['largest']}"
                             for k, v in lang["families"].items()})
print("\nmost phishing-marking words:", [w["term"] for w in lang["log_odds"]["phishing"]])
print("most ham-marking words     :", [w["term"] for w in lang["log_odds"]["ham"]])

Gradient boosting trên token, dấu, độ dài và sáu đặc điểm hình thức đạt khoảng 0,88. Đó là mức một mô hình đạt được khi chỉ nhìn hình thức
tin, không đọc nghĩa. Riêng bản "strict" (bỏ hẳn token, chỉ giữ dấu, độ dài và hình thức) cho biết phần nào còn lại khi không dùng tới cách
nhà phát hành che dữ liệu.

Ô dưới gom mọi thứ vào một kết quả, so với `shallow_cues.json`, rồi ghi dự đoán của hai sàn gradient boosting trên tập test ra file tạm
(Bước 13 dùng file này) và so từng byte với `floor_test_predictions.csv`.

In [ ]:
shallow = {"n": dict(tot), "accented_pct": accented, "language": lang,
           "any_token_pct": {l: round(100 * any_tok[l] / tot[l], 1) for l in labs}, "token_pct": per_token,
           "median_chars": {l: int(statistics.median(lens[l])) for l in labs}, "baselines": base,
           "split": {"train": len(train_r), "test": len(test_r)}, "registered": False, "seed": 0}
check_json("4 shallow", "shallow_cues.json", shallow, SAVED / "shallow_cues.json")

full_floor = gbm_predict(tr_t, te_t, lambda r: f_all(r) + format_feats(r))
strict_floor = gbm_predict(tr_t, te_t, lambda r: f_acc(r) + f_len(r) + format_feats(r))
FLOOR_FILE = TMP / "floor_test_predictions.csv"
with open(FLOOR_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["message_id", "text_sha1", "label", "floor_full", "floor_strict"])
    for r, a_, b_ in zip(te_t, full_floor, strict_floor):
        w.writerow([r["message_id"], r["text_sha1"], r["label"], int(a_), int(b_)])
check("4 shallow", "floor_test_predictions.csv byte-identical", same_file(FLOOR_FILE, SAVED / "floor_test_predictions.csv"), True)
report("4 shallow")

## Bước 5: Nhãn bám theo lô thu thập tới mức nào?

Bảng ở Bước 1 cho thấy hai lớp không trải đều qua các lô. Hai lô ngày 22/07/2026 và 24/07/2026 chứa phần lớn tin phishing và gần như không
có ham. Khi chia ngẫu nhiên (kể cả chia theo hash nội dung), tin của hai lô này nằm ở cả train lẫn test. Mô hình có thể đạt điểm cao chỉ nhờ
nhận ra "tin này thuộc hai lô đó", chứ không phải nhờ nhận ra dấu hiệu lừa đảo.

Bước này dùng hai mô hình: "probe" là TF-IDF + LogReg như Bước 6, và "text arm" là MLP trên embedding PhoBERT như Bước 8. Đơn vị là "nhóm gần
trùng": các tin giống nhau từ 0,90 trở lên được gom thành một nhóm và luôn đi cùng nhau, để không tin nào trong tập chấm có một tin gần như
y hệt trong tập train. Ngoài ra, 20% số nhóm ham được giữ riêng làm tập ham cố định để chấm mọi thí nghiệm.

Cách đọc các con số:
- recall ở ngưỡng 0,5: tỷ lệ tin phishing bị bắt khi báo "phishing" mỗi lần xác suất từ 0,5 trở lên.
- ROC-AUC: mô hình có xếp tin phishing cao hơn tin ham không, bất kể ngưỡng (1 là hoàn hảo, 0,5 là đoán bừa).
- recall ở FPR 2%: chọn ngưỡng sao cho đúng 2% tin ham bị báo nhầm, rồi đếm tỷ lệ phishing bắt được. Cách này công bằng hơn ngưỡng 0,5 khi
  so các mô hình huấn luyện trên dữ liệu có tỷ lệ lớp khác nhau.

Các thí nghiệm:
1. Bỏ hai lô: bỏ hẳn hai lô khỏi train rồi chấm trên tin phishing của chính hai lô đó. Đối chứng: thay vì bỏ hai lô, bỏ ngẫu nhiên các nhóm
   phishing cho tới khi số tin phishing còn lại để train bằng lúc bỏ hai lô (5 lần). Nếu bỏ hai lô làm điểm tụt sâu hơn hẳn đối chứng, thì
   điểm cao trước đó chủ yếu nhờ nhận ra lô.
2. Chiều ngược lại: chỉ dùng tin phishing của hai lô để train, rồi chấm trên tin phishing ngoài hai lô, kèm đối chứng tương tự.
3. Lần lượt để riêng từng lô ra làm tập chấm, so với chia 5 phần theo nhóm gần trùng và với "lô giả" có cùng số tin mỗi lớp nhưng gán ngẫu
   nhiên. Có thêm bản LogReg cân bằng lớp, và bản mỗi mô hình có ngưỡng 2% riêng.
4. Huấn luyện trên các lô gần như thuần một lớp, chấm trên các lô lẫn hai lớp.
5. Vài con số mô tả: trần của kênh URL, tin test chỉ khác tin train ở khoảng trắng hay dấu câu, tiền tố của `message_id`.

Kết quả đã lưu được tính với BLAS một luồng, nên bước này chạy trong `one_thread()`. Ô đầu tính phần mô tả, ô sau chạy toàn bộ thí nghiệm
(khoảng 1 phút).

In [ ]:
POS_BATCHES = ("22/07/2026", "24/07/2026")
T_all = np.array(texts, dtype=object)
date = np.array([r["date"] for r in rows])
sha_ = np.array([r["text_sha1"] for r in rows])
acc = np.array([int(r["accented"]) for r in rows])
inb = np.isin(date, POS_BATCHES)

print(f"the two batches hold {int((inb & (y == 1)).sum())} of {int(y.sum())} phishing rows and {int((inb & (y == 0)).sum())} ham row(s)")
print("accented (%):", {"phishing in the two batches": round(100 * acc[inb & (y == 1)].mean(), 1),
                        "phishing elsewhere": round(100 * acc[~inb & (y == 1)].mean(), 1), "ham": round(100 * acc[y == 0].mean(), 1)})

Tin phishing trong hai lô có dấu khoảng 87%, còn tin phishing ngoài hai lô chỉ khoảng 36%, gần với ham (khoảng 39%). Như vậy nhận xét "phishing
viết có dấu nhiều hơn" là đặc điểm của hai lô này, không phải của tin phishing nói chung.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupKFold

def probe():
    return make_pipeline(TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=3000, C=4, random_state=0))

def bal_probe():
    return make_pipeline(TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=3000, C=4, random_state=0, class_weight="balanced"))

def text_arm(Xtr, ytr, seed=0):
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(128,), max_iter=400, random_state=seed,
                                                         early_stopping=True, n_iter_no_change=15)).fit(Xtr, ytr)

def near_dup_components(txts, sim=0.90):
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True).fit_transform(txts)
    return connected_components(csr_matrix(cosine_similarity(X) >= sim), directed=False)[1]

def scores(yev, sc):
    # Recall at 0.5, ROC-AUC, and recall at the threshold that flags 2% of the evaluation ham
    thr = np.quantile(sc[yev == 0], 0.98)
    return {"recall_at_0.5": round(float((sc[yev == 1] >= 0.5).mean()), 3), "roc_auc": round(float(roc_auc_score(yev, sc)), 3),
            "recall_at_2pct_fpr": round(float((sc[yev == 1] > thr).mean()), 3)}

def norm_ws(t):
    return " ".join(t.split())

def norm_loose(t):
    t = unicodedata.normalize("NFC", t).lower()
    return " ".join(re.sub(r"[^\w\s]", " ", t).split())

def batch_audit(emb, fusion_diag):
    out = {"registered": False, "status": "post-hoc, descriptive"}
    # batch x class
    tab = collections.defaultdict(lambda: [0, 0])
    for d, l in zip(date, y):
        tab[d or "(undated)"][l] += 1
    batches = {d: {"ham": v[0], "positive": v[1], "pos_share": round(v[1] / sum(v), 3)} for d, v in tab.items()}
    pure = {d for d, v in batches.items() if max(v["pos_share"], 1 - v["pos_share"]) >= 0.95}
    out.update(batches=batches, n_batches=len(batches), pure_threshold=0.95,
               rows_in_pure_batches=int(sum(batches[d]["ham"] + batches[d]["positive"] for d in pure)))
    out["positive_only_batches"] = {"dates": list(POS_BATCHES), "positive": int((inb & (y == 1)).sum()),
                                    "ham": int((inb & (y == 0)).sum()), "positive_total": int((y == 1).sum())}
    out["accented_pct"] = {"positive_in_batches": round(100 * acc[inb & (y == 1)].mean(), 1),
                           "positive_outside": round(100 * acc[~inb & (y == 1)].mean(), 1),
                           "positive_outside_n": int((~inb & (y == 1)).sum()),
                           "ham": round(100 * acc[y == 0].mean(), 1), "positive_all": round(100 * acc[y == 1].mean(), 1)}

    # leave the two batches out, against controls matched on positive training rows
    first = {}
    for i, h in enumerate(sha_):
        first.setdefault(h, i)
    utxt = sorted(first)
    comp_of = dict(zip(utxt, near_dup_components([T_all[first[h]] for h in utxt])))
    lab_of = {h: int(y[first[h]]) for h in utxt}
    ham_comps = sorted({comp_of[h] for h in utxt if lab_of[h] == 0})
    eval_ham_comps = set(np.random.default_rng(0).choice(ham_comps, int(0.2 * len(ham_comps)), replace=False).tolist())
    eval_ham = [h for h in utxt if lab_of[h] == 0 and comp_of[h] in eval_ham_comps]
    row_comp = np.array([comp_of[h] for h in sha_])

    def run(held_pos, extra_excl_rows=None, arm_seeds=(0, 1, 2)):
        held_c = {comp_of[h] for h in held_pos} | eval_ham_comps
        trm = ~np.isin(row_comp, list(held_c))
        if extra_excl_rows is not None:
            trm &= ~extra_excl_rows
        pos_eval = [h for h in held_pos if lab_of[h] == 1]
        ev = [first[h] for h in pos_eval] + [first[h] for h in eval_ham]
        yev = np.array([1] * len(pos_eval) + [0] * len(eval_ham))
        res_ = {"train_positive_rows": int((trm & (y == 1)).sum()), "train_rows": int(trm.sum()),
                "eval_positive_texts": len(pos_eval), "eval_ham_texts": len(eval_ham)}
        res_["probe"] = scores(yev, probe().fit(T_all[trm], y[trm]).predict_proba(T_all[ev])[:, 1])
        arm = [scores(yev, text_arm(emb[trm], y[trm], s_).predict_proba(emb[ev])[:, 1]) for s_ in arm_seeds]
        res_["text_arm"] = {k: round(float(np.mean([a_[k] for a_ in arm])), 3) for k in arm[0]}
        return res_

    held_txt = sorted({h for h in sha_[inb & (y == 1)]})
    batch = run(held_txt, extra_excl_rows=inb)
    target = batch["train_positive_rows"]
    pos_comps = sorted({comp_of[h] for h in utxt if lab_of[h] == 1} - eval_ham_comps)
    controls = []
    for s_ in range(5):
        held_c = set()
        for c in np.random.default_rng(100 + s_).permutation(pos_comps):
            held_c.add(c)
            if int((~np.isin(row_comp, list(held_c | eval_ham_comps)) & (y == 1)).sum()) <= target:
                break
        controls.append(run([h for h in utxt if comp_of[h] in held_c and lab_of[h] == 1]))
    span = lambda key, m_: [round(min(c[key][m_] for c in controls), 3), round(max(c[key][m_] for c in controls), 3)]
    out["leave_batches_out"] = {
        "unit": "near-duplicate component (char 3-5-gram TF-IDF, cosine >= 0.9)", "eval_ham_texts": len(eval_ham),
        "batch": batch, "controls": controls,
        "control_span": {k: {m_: span(k, m_) for m_ in ("recall_at_0.5", "roc_auc", "recall_at_2pct_fpr")} for k in ("probe", "text_arm")},
        "control_train_positive_rows": [c["train_positive_rows"] for c in controls]}

    # the reverse direction
    other_pos = sorted({h for h in sha_[~inb & (y == 1)]} - set(held_txt))
    def score_held(held_pos):
        held_c = {comp_of[h] for h in held_pos} | eval_ham_comps
        trm = ~np.isin(row_comp, list(held_c))
        ev = [first[h] for h in held_pos] + [first[h] for h in eval_ham]
        yev = np.array([1] * len(held_pos) + [0] * len(eval_ham))
        return int((trm & (y == 1)).sum()), scores(yev, probe().fit(T_all[trm], y[trm]).predict_proba(T_all[ev])[:, 1])
    n_tr, sc_rev = score_held(other_pos)
    rev_ctl = []
    for s_ in range(5):
        held_c, hp = set(), []
        for c in np.random.default_rng(200 + s_).permutation(pos_comps):
            held_c.add(c)
            hp = [h for h in utxt if comp_of[h] in held_c and lab_of[h] == 1]
            if len(hp) >= len(other_pos):
                break
        n_c, sc_c = score_held(hp)
        rev_ctl.append({"train_positive_rows": n_c, "eval_positive_texts": len(hp), "probe": sc_c})
    out["reverse"] = {"train_positive_rows": n_tr, "eval_positive_texts": len(other_pos), "probe": sc_rev, "controls": rev_ctl,
                      "control_span": {m_: [round(min(c["probe"][m_] for c in rev_ctl), 3), round(max(c["probe"][m_] for c in rev_ctl), 3)]
                                       for m_ in ("roc_auc", "recall_at_2pct_fpr")}}

    # every batch held out in turn, against folds grouped by near-duplicate component and pseudo-batches
    Tu = np.array([T_all[first[h]] for h in utxt], dtype=object)
    Yu = np.array([lab_of[h] for h in utxt])
    B0 = np.array([date[first[h]] or "(undated)" for h in utxt])
    Cu = np.array([comp_of[h] for h in utxt])
    maj = {c: collections.Counter(B0[Cu == c]).most_common(1)[0][0] for c in set(Cu.tolist())}
    Bu = np.array([maj[c] for c in Cu])
    two = np.isin(B0, POS_BATCHES)

    def pooled(grp, k, model=probe):
        oof = np.zeros(len(Yu))
        for a_, b_ in GroupKFold(k).split(Tu, Yu, grp):
            oof[b_] = model().fit(Tu[a_], Yu[a_]).predict_proba(Tu[b_])[:, 1]
        thr = np.quantile(oof[Yu == 0], 0.98)
        pred = (oof >= 0.5).astype(int)
        return {"roc_auc": round(float(roc_auc_score(Yu, oof)), 3), "f1_at_0.5": round(float(f1_score(Yu, pred)), 3),
                "precision_at_0.5": round(float(precision_score(Yu, pred)), 3), "recall_at_0.5": round(float(recall_score(Yu, pred)), 3),
                "ham_fpr_at_0.5": round(float(pred[Yu == 0].mean()), 3), "recall_at_2pct_fpr": round(float((oof[Yu == 1] > thr).mean()), 3),
                "recall_at_2pct_fpr_two_batches": round(float((oof[(Yu == 1) & two] > thr).mean()), 3),
                "recall_at_2pct_fpr_other": round(float((oof[(Yu == 1) & ~two] > thr).mean()), 3)}

    def per_model(grp, k):
        ev_ham = np.isin(Cu, list(eval_ham_comps)) & (Yu == 0)
        hit = np.zeros(len(Yu), dtype=bool)
        for a_, b_ in GroupKFold(k).split(Tu, Yu, grp):
            a_ = a_[~ev_ham[a_]]
            m_ = probe().fit(Tu[a_], Yu[a_])
            thr = np.quantile(m_.predict_proba(Tu[ev_ham])[:, 1], 0.98)
            b_ = b_[Yu[b_] == 1]
            if len(b_):
                hit[b_] = m_.predict_proba(Tu[b_])[:, 1] > thr
        return {"recall_at_2pct_fpr": round(float(hit[Yu == 1].mean()), 3),
                "recall_at_2pct_fpr_two_batches": round(float(hit[(Yu == 1) & two].mean()), 3)}

    def pseudo_batches(seed):
        rng = np.random.default_rng(300 + seed)
        target_ = {g: [int(((Bu == g) & (Yu == 0)).sum()), int(((Bu == g) & (Yu == 1)).sum())] for g in sorted(set(Bu.tolist()))}
        comps = list(set(Cu.tolist()))
        rng.shuffle(comps)
        out_g = np.empty(len(Yu), dtype=object)
        for c in comps:
            idx = np.where(Cu == c)[0]
            lab = int(round(Yu[idx].mean()))
            g = max(target_, key=lambda g_: (target_[g_][lab], rng.random()))
            target_[g][lab] -= len(idx)
            out_g[idx] = g
        return out_g

    E = np.array([emb[first[h]] for h in utxt])
    def arm_lobo(grp, k):
        ev_ham = np.isin(Cu, list(eval_ham_comps)) & (Yu == 0)
        oof = np.zeros(len(Yu)); hit = np.zeros(len(Yu), dtype=bool)
        for a_, b_ in GroupKFold(k).split(E, Yu, grp):
            oof[b_] = text_arm(E[a_], Yu[a_], 0).predict_proba(E[b_])[:, 1]
            a2 = a_[~ev_ham[a_]]
            m2 = text_arm(E[a2], Yu[a2], 0)
            thr2 = np.quantile(m2.predict_proba(E[ev_ham])[:, 1], 0.98)
            bp = b_[Yu[b_] == 1]
            if len(bp):
                hit[bp] = m2.predict_proba(E[bp])[:, 1] > thr2
        thr = np.quantile(oof[Yu == 0], 0.98)
        return {"roc_auc": round(float(roc_auc_score(Yu, oof)), 3), "recall_at_2pct_fpr": round(float((oof[Yu == 1] > thr).mean()), 3),
                "recall_at_2pct_fpr_per_model": round(float(hit[Yu == 1].mean()), 3)}
    arm_b, arm_c = arm_lobo(Bu, len(set(Bu))), arm_lobo(Cu, 5)

    oof_b = np.zeros(len(Yu))
    for a_, b_ in GroupKFold(len(set(Bu))).split(Tu, Yu, Bu):
        oof_b[b_] = probe().fit(Tu[a_], Yu[a_]).predict_proba(Tu[b_])[:, 1]
    per_batch = {g: {"auc": round(float(roc_auc_score(Yu[Bu == g], oof_b[Bu == g])), 3), "positive_texts": int(Yu[Bu == g].sum()),
                     "ham_texts": int((Yu[Bu == g] == 0).sum())}
                 for g in sorted(set(Bu.tolist())) if 0 < Yu[Bu == g].sum() < (Bu == g).sum()}
    pseudo, pseudo_bal, pseudo_pm = [], [], []
    for s_ in range(5):
        pg = pseudo_batches(s_)
        k_ = len(set(pg.tolist()))
        pseudo.append(pooled(pg, k_)); pseudo_bal.append(pooled(pg, k_, bal_probe)); pseudo_pm.append(per_model(pg, k_))
    out["leave_one_batch_out"] = {
        "by_batch": pooled(Bu, len(set(Bu))), "by_component_5fold": pooled(Cu, 5),
        "by_batch_balanced": pooled(Bu, len(set(Bu)), bal_probe), "by_component_5fold_balanced": pooled(Cu, 5, bal_probe),
        "by_batch_per_model": per_model(Bu, len(set(Bu))), "by_component_5fold_per_model": per_model(Cu, 5),
        "pseudo_batches": pseudo,
        "pseudo_span": {m_: [round(min(p_[m_] for p_ in pseudo), 3), round(max(p_[m_] for p_ in pseudo), 3)]
                        for m_ in ("roc_auc", "recall_at_2pct_fpr")},
        "pseudo_balanced_span": [round(min(p_["recall_at_2pct_fpr"] for p_ in pseudo_bal), 3),
                                 round(max(p_["recall_at_2pct_fpr"] for p_ in pseudo_bal), 3)],
        "pseudo_per_model_span": [round(min(p_["recall_at_2pct_fpr"] for p_ in pseudo_pm), 3),
                                  round(max(p_["recall_at_2pct_fpr"] for p_ in pseudo_pm), 3)],
        "text_arm_by_batch": arm_b, "text_arm_by_component_5fold": arm_c, "per_batch_auc_mixed": per_batch,
        "batches": len(set(Bu)), "undated_is_one_group": True,
        "components_spanning_batches": int(sum(1 for c in set(Cu.tolist()) if len(set(B0[Cu == c])) > 1)),
        "unit": "distinct text; near-duplicate components kept whole"}

    nurl = np.array([int(r["n_urls"]) for r in rows])
    out["no_url_positive"] = {"total": int(((y == 1) & (nurl == 0)).sum()), "in_two_batches": int(((y == 1) & (nurl == 0) & inb).sum())}
    is_pure = np.isin(date, list(pure))
    tr_p = is_pure & ~np.isin(sha_, list(set(sha_[~is_pure])))
    te_m = ~is_pure
    p = probe().fit(T_all[tr_p], y[tr_p]).predict(T_all[te_m])
    out["pure_to_mixed"] = {"train_rows": int(tr_p.sum()), "test_rows": int(te_m.sum()), "test_positive": int(y[te_m].sum()),
                            "f1": round(float(f1_score(y[te_m], p)), 3), "precision": round(float(precision_score(y[te_m], p)), 3),
                            "recall": round(float(recall_score(y[te_m], p)), 3)}
    pos_url = fusion_diag["url/has_url"]["n_pos"]
    pos_all = pos_url + fusion_diag["url/no_url"]["n_pos"]
    r_max = pos_url / pos_all
    out["url_ceiling"] = {"test_positive": pos_all, "test_positive_with_url": pos_url, "test_positive_no_url_pct": round(100 * (1 - r_max), 1),
                          "max_recall": round(r_max, 3), "max_f1": round(2 * r_max / (1 + r_max), 3),
                          "note": "perfect precision, every URL-bearing positive caught"}
    twins = {}
    for name, fn in (("whitespace", norm_ws), ("case_punct", norm_loose)):
        trn = {fn(t) for t in T_all[tr_mask]}
        hit = [i for i in np.where(te_mask)[0] if fn(T_all[i]) in trn]
        twins[name] = {"test_rows": len(hit), "ham": int((y[hit] == 0).sum()), "positive": int((y[hit] == 1).sum())}
    grp_ws = collections.defaultdict(set)
    for t, l in zip(T_all, y):
        grp_ws[norm_ws(t)].add(int(l))
    twins["cross_label_whitespace_groups"] = sum(1 for v in grp_ws.values() if len(v) == 2)
    out["twins"] = twins
    aug = np.array([d.endswith("/8/2026") or d.endswith("/08/2026") for d in date])
    out["august_rows"] = {"train": int((aug & tr_mask).sum()), "test": int((aug & te_mask).sum())}
    pre = collections.defaultdict(lambda: [0, 0])
    for r, l in zip(rows, y):
        pre[r["message_id"].split("_")[0]][l] += 1
    out["id_prefix"] = {k: {"ham": v[0], "positive": v[1]} for k, v in sorted(pre.items())}
    return out

Hàm `batch_audit` cần hai thứ chưa có: embedding PhoBERT (Bước 8) và số tin phishing có/không có URL trong test (cũng Bước 8, dùng cho trần
của kênh URL). Vì vậy notebook chạy nó ở cuối Bước 8, rồi quay lại đọc kết quả ở đó.

## Bước 6: Baseline đơn giản nhất

Mô hình gồm TF-IDF trên từ đơn và cặp từ liền nhau (1-gram và 2-gram), cộng LogReg với C = 4. Không cần bộ tách từ tiếng Việt, không cần
GPU. Đây là mốc để xem mô hình transformer có đáng dùng không. Ngoài điểm số, ô dưới in các từ có trọng số lớn nhất về mỗi lớp (hệ số của
LogReg) để thấy mô hình học được gì.

In [ ]:
Xtr_t = [t for t, k in zip(texts, tr_mask) if k]
Xte_t = [t for t, k in zip(texts, te_mask) if k]
pipe = probe().fit(Xtr_t, y[tr_mask])
score = pipe.predict_proba(Xte_t)[:, 1]
pred = (score > 0.5).astype(int)
lin = {"accuracy": round(float((pred == y[te_mask]).mean()), 4), "precision": round(float(precision_score(y[te_mask], pred, zero_division=0)), 4),
       "recall": round(float(recall_score(y[te_mask], pred)), 4), "f1": round(float(f1_score(y[te_mask], pred)), 4),
       "roc_auc": round(float(roc_auc_score(y[te_mask], score)), 4), "pr_auc": round(float(average_precision_score(y[te_mask], score)), 4),
       "features": int(len(pipe.steps[0][1].vocabulary_)), "n_train": int(tr_mask.sum()), "n_test": int(te_mask.sum())}
print(lin)
vocab = np.array(pipe.steps[0][1].get_feature_names_out())
order = np.argsort(pipe.steps[1][1].coef_[0])
print("\nstrongest ham terms     :", list(vocab[order[:15]]))
print("strongest phishing terms:", list(vocab[order[-15:][::-1]]))

tl = saved("text_linear.json")
for k, v in lin.items():
    check("6 linear", k, v, tl[k])
report("6 linear")

`text_linear.json` còn có `size_mb` và `host_us_per_message` (kích thước file mô hình và thời gian chạy trên máy). Hai trường này phụ thuộc
máy nên notebook không so.

Cách đọc danh sách từ: TF-IDF chuyển hết về chữ thường và bỏ dấu ngoặc, nên token che hiện ra là `tb`, `qc`, `number`, `date`, `time`, và
tất cả đều nghiêng về phía ham. Phía ham còn có cụm không dấu quen thuộc của tin nhà mạng (`quy khach`, `soan`, `gui`). Phía phishing có mảnh
URL (`http`, `com`, `top`) và từ có dấu (`bạn`, `nhận`). Bước 14 bỏ hết dấu để xem cách viết có dấu hay không dấu gánh bao nhiêu phần tín hiệu.

## Bước 7: Kênh URL

Từ bước này là phép so sánh đã đăng ký trước trong `PREREG_smishing.md`. Kênh URL được dựng qua ba bước:
1. `first_url(text)` lấy URL đầu tiên trong tin theo luật của bản đăng ký (mọi chỗ khớp regex), thêm `http://` nếu thiếu.
2. `extract(url)` tính các đặc trưng từ chuỗi URL theo bộ CompPhish: độ dài, số chữ số, số dấu chấm, độ hỗn loạn ký tự (entropy)... Mọi phép
   đếm làm trên URL đã bỏ phần `http://` hay `https://`, để phần đầu này không lọt vào đặc trưng.
3. Giữ đúng 21 cột `COMPPHISH`, cùng bộ đặc trưng các bài về URL web dùng. Cột `is_https` bị bỏ vì nó phản ánh cách thu thập dữ liệu chứ không
   phải tín hiệu lừa đảo.

Tin không có URL nhận một vector toàn số 0. Đây là một lý do kênh URL yếu: khoảng 37% tin phishing không có URL nào.

In [ ]:
from collections import Counter
from math import log2
from urllib.parse import urlparse

_EXT = tldextract.TLDExtract(suffix_list_urls=())  # ICANN suffixes only, bundled snapshot
_IP_RE = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")
COMPPHISH = ["url_len", "dom_len", "is_ip", "tld_len", "subdom_cnt", "letter_cnt", "digit_cnt", "special_cnt", "eq_cnt", "qm_cnt",
             "amp_cnt", "dot_cnt", "dash_cnt", "under_cnt", "letter_ratio", "digit_ratio", "spec_ratio", "slash_cnt", "entropy",
             "path_len", "query_len"]
URL_RE = re.compile(r"(?:https?://)?((?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+[a-z]{2,})(?:/[^\s,;)\]]*)?", re.I)

def _entropy(s):
    if not s:
        return 0.0
    n = len(s)
    return -sum((c / n) * log2(c / n) for c in Counter(s).values())

def extract(url):
    url = str(url).strip()
    with_scheme = url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url
    p = urlparse(with_scheme)
    host = (p.hostname or "").lower()
    core = re.sub(r"^[a-z]+://", "", url, flags=re.I)  # every lexical count ignores the scheme
    ext = _EXT(with_scheme)
    suffix, dom, subdom = ext.suffix, ext.registered_domain or host, ext.subdomain
    url_len = len(core)
    letter_cnt = sum(c.isalpha() for c in core); digit_cnt = sum(c.isdigit() for c in core)
    special_cnt = url_len - letter_cnt - digit_cnt
    return {"url_len": url_len, "dom_len": len(dom), "is_ip": 1 if _IP_RE.match(host) else 0, "tld_len": len(suffix),
            "subdom_cnt": len([s_ for s_ in subdom.split(".") if s_]) if subdom else 0,
            "letter_cnt": letter_cnt, "digit_cnt": digit_cnt, "special_cnt": special_cnt,
            "eq_cnt": core.count("="), "qm_cnt": core.count("?"), "amp_cnt": core.count("&"), "dot_cnt": core.count("."),
            "dash_cnt": core.count("-"), "under_cnt": core.count("_"),
            "letter_ratio": letter_cnt / url_len if url_len else 0.0, "digit_ratio": digit_cnt / url_len if url_len else 0.0,
            "spec_ratio": special_cnt / url_len if url_len else 0.0, "slash_cnt": core.count("/"), "entropy": _entropy(core),
            "path_len": len(p.path.strip("/")), "query_len": len(p.query)}

def first_url(text, rule="registered"):
    for m in URL_RE.finditer(text or ""):
        if rule == "psl" and not has_public_suffix(m.group(1).lower()):
            continue
        u = m.group(0)
        return u if u.lower().startswith("http") else "http://" + u
    return ""

def url_matrix(urls_):
    X = np.zeros((len(urls_), len(COMPPHISH)), dtype=np.float32)
    for i, u in enumerate(urls_):
        if u:
            f = extract(u)
            X[i] = [float(f.get(c, 0) or 0) for c in COMPPHISH]
    return X

u = first_url("Vui long truy cap vietcombank.vn-gll.top/login?id=1 ngay")
print("first URL:", u)
display(pd.Series(extract(u)).to_frame("value").T)

urls = [first_url(t) for t in texts]
has_url = np.array([bool(x) for x in urls])
Xurl = url_matrix(urls)
print(f"messages with a URL (registered rule): {has_url.sum()} / {len(texts)}")
print(f"share with a URL  ham: {100 * has_url[y == 0].mean():.1f}%   phishing: {100 * has_url[y == 1].mean():.1f}%")

Con số này lớn hơn số tin có URL ở phần mô tả corpus (trường `with_url` ở Bước 2). Lý do là kênh URL dùng luật của bản đăng
ký, coi mọi chỗ khớp regex là URL, kể cả mẩu như `tp.hcm`. Phần mô tả corpus thì chỉ đếm URL có đuôi thật. Bước 11 làm lại với luật chặt hơn
(`psl`) để xem kết luận có đổi không.

## Bước 8: Kênh text (PhoBERT), kênh fusion, và ba kênh × 10 seed

Kênh text dùng PhoBERT-base mà không huấn luyện thêm (frozen). PhoBERT đọc một tin và trả về một vector 768 số cho mỗi token. Để có một vector
duy nhất cho cả tin, lấy trung bình các vector token, bỏ qua các token đệm (padding) thêm vào cho đủ độ dài; phép này gọi là mean pooling theo
attention mask. Tin dài hơn 256 token bị cắt bớt.

Mô hình được tải đúng revision đã ghim. Trọng số được đọc từ file gốc `pytorch_model.bin` và nạp bằng tay, vì file `safetensors` mà Hugging Face
tự chuyển đổi cho repo này thiếu ma trận embedding của từ: nạp từ file đó vẫn chạy nhưng cho ra vector sai mà không báo lỗi. Hàm `phobert()`
kiểm rằng không trọng số nào bị thiếu.

Tính embedding cho 2.991 tin mất khoảng 2 phút trên CPU, nên kết quả được lưu trong `sms_work/emb_full.npy`, kèm "vân tay" SHA-256 của toàn bộ
nội dung tin. Lần chạy sau, nếu vân tay khớp thì dùng lại, còn nếu lệch thì tính lại.

Kênh fusion nối hai vector lại: 768 + 21 = 789 số.

Cả ba kênh dùng chung `fit_score`: chuẩn hoá các cột (`StandardScaler`), rồi huấn luyện một MLP có một lớp ẩn 128 nơ-ron, tối đa 400 vòng, tự
dừng sớm khi điểm trên một phần train giữ lại không cải thiện nữa (early stopping). Điểm là F1 của lớp phishing ở ngưỡng 0,5. Bài chạy 10 seed;
mỗi seed chỉ đổi cách khởi tạo MLP, cách chia train/test giữ nguyên.

In [ ]:
SEEDS = list(range(10))

_PHOBERT = {}

def phobert():
    # PhoBERT-base at the pinned revision. The weights are read from the original pytorch_model.bin and loaded by
    # hand: the automatically converted safetensors file on the Hub lacks the word-embedding matrix, and a model
    # loaded from it silently returns different vectors.
    if not _PHOBERT:
        import torch
        from transformers import AutoTokenizer, RobertaConfig, RobertaModel
        cfg = RobertaConfig.from_pretrained(PHOBERT_REPO, revision=PHOBERT_REV)
        sd = torch.load(hf_hub_download(PHOBERT_REPO, "pytorch_model.bin", revision=PHOBERT_REV), map_location="cpu", weights_only=True)
        sd = {k.removeprefix("roberta."): v for k, v in sd.items() if k.startswith("roberta.")}
        model = RobertaModel(cfg, add_pooling_layer=False)
        missing, _ = model.load_state_dict(sd, strict=False)
        assert not missing, f"weights missing from the checkpoint: {missing}"
        _PHOBERT.update(tok=AutoTokenizer.from_pretrained(PHOBERT_REPO, revision=PHOBERT_REV), model=model.eval())
    return _PHOBERT["tok"], _PHOBERT["model"]

def embed(txts, name):
    # Mean-pooled PhoBERT vectors, cached in sms_work/ under a fingerprint of the exact input texts
    h = hashlib.sha256()
    for t in txts:
        b_ = (t or "").encode("utf-8", "surrogatepass")
        h.update(len(b_).to_bytes(8, "big"))
        h.update(b_)
    cache, stamp = WORK / f"emb_{name}.npy", WORK / f"emb_{name}.sha256"
    if cache.exists() and stamp.exists() and stamp.read_text() == h.hexdigest():
        return np.load(cache)
    import torch
    tok_, mdl = phobert()
    out = []
    with torch.no_grad():
        for i in range(0, len(txts), 32):
            b_ = tok_(txts[i:i + 32], padding=True, truncation=True, max_length=256, return_tensors="pt")
            hdn = mdl(**b_).last_hidden_state
            mask = b_["attention_mask"].unsqueeze(-1).float()
            out.append(((hdn * mask).sum(1) / mask.sum(1).clamp(min=1)).numpy())
    e = np.vstack(out).astype(np.float32)
    np.save(cache, e)
    stamp.write_text(h.hexdigest())
    return e

def fit_score(Xtr, ytr, Xte, yte, seed):
    sc = StandardScaler().fit(Xtr)
    clf = MLPClassifier(hidden_layer_sizes=(128,), max_iter=400, random_state=seed, early_stopping=True, n_iter_no_change=15)
    clf.fit(sc.transform(Xtr), ytr)
    p = clf.predict(sc.transform(Xte)); prob = clf.predict_proba(sc.transform(Xte))[:, 1]
    tp = int(((yte == 1) & (p == 1)).sum()); fp = int(((yte == 0) & (p == 1)).sum())
    fn = int(((yte == 1) & (p == 0)).sum()); tn = int(((yte == 0) & (p == 0)).sum())
    return {"f1": f1_score(yte, p, pos_label=1, zero_division=0), "precision": precision_score(yte, p, pos_label=1, zero_division=0),
            "recall": recall_score(yte, p, pos_label=1, zero_division=0), "fpr": fp / max(fp + tn, 1),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn, "roc_auc": roc_auc_score(yte, prob), "pr_auc": average_precision_score(yte, prob)}, p

def run_arms(arm_X, trm, tem, seeds=SEEDS, verbose=False):
    # Fit every arm for every seed; return per-seed metrics and predictions
    r_, p_ = {k: [] for k in arm_X}, {k: [] for k in arm_X}
    for s_ in seeds:
        for k, X in arm_X.items():
            met, p = fit_score(X[trm], y[trm], X[tem], y[tem], s_)
            r_[k].append(met); p_[k].append(p)
        if verbose:
            print(f"seed {s_}: " + "  ".join(f"{k}={r_[k][-1]['f1']:.3f}" for k in arm_X))
    return r_, p_

Xtxt = embed(texts, "full")
Xfus = np.hstack([Xtxt, Xurl])
print("text:", Xtxt.shape, "| url:", Xurl.shape, "| fusion:", Xfus.shape)
arms = {"url": Xurl, "text": Xtxt, "fusion": Xfus}
res, preds = run_arms(arms, tr_mask, te_mask, verbose=True)
mean4 = lambda v, key: round(float(np.mean([x[key] for x in v])), 4)
pd.DataFrame({k: {m_: mean4(v, m_) for m_ in ("f1", "precision", "recall", "fpr", "roc_auc", "pr_auc")} for k, v in res.items()}).T

Đọc kết quả:
- T1 (fusion trừ URL): bản đăng ký coi là thành công nếu chênh từ 3 điểm F1 trở lên. Thực tế chênh khoảng 32 điểm, nên T1 đạt.
- T2 (text trừ URL): bản đăng ký dự đoán URL hơn text ít nhất 5 điểm. Thực tế text hơn URL khoảng 32 điểm, tức dự đoán sai. Bài báo cáo thẳng
  kết quả âm này, vì nó đã được đăng ký trước.
- fusion trừ text (post-hoc): gần bằng 0. Khi đã có text, thêm URL gần như không giúp gì.

Kênh URL có precision cao (khoảng 0,87) nhưng recall thấp (khoảng 0,47): khi nó báo phishing thì thường đúng, nhưng nó bỏ sót hơn nửa số tin
phishing.

### Kết quả theo nhóm tin và file phiếu bầu

Bản đăng ký yêu cầu báo thêm kết quả trên ba nhóm: tin không có URL, tin có URL, và tin phishing có link rút gọn. Ngoài ra còn một bảng "phiếu
bầu": với mỗi tin test, mỗi kênh có bao nhiêu seed (trong 10) báo tin đó là phishing. Bước 13 dùng bảng này để phân tích lỗi.

In [ ]:
short_ids = {r["message_id"] for r in rows if int(r["n_shortened"]) > 0 and r["label"] == "1"}
strata = {"no_url": ~has_url[te_mask], "has_url": has_url[te_mask],
          "shortened_phish": np.array([rows[i]["message_id"] in short_ids for i in te_idx])}
diag = {}
for name in arms:
    for tag, mask in strata.items():
        if mask.sum():
            diag[f"{name}/{tag}"] = {
                "n": int(mask.sum()), "n_pos": int(y[te_mask][mask].sum()),
                "f1": round(float(np.mean([f1_score(y[te_mask][mask], p[mask], pos_label=1, zero_division=0) for p in preds[name]])), 4),
                "detected_mean": round(float(np.mean([int(((y[te_mask][mask] == 1) & (p[mask] == 1)).sum()) for p in preds[name]])), 2)}
display(pd.DataFrame(diag).T)

VOTES_FILE = TMP / "fusion_test_predictions.csv"
with open(VOTES_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["message_id", "text_sha1", "label", "has_url"] + [f"{k}_votes" for k in arms])
    for j, i in enumerate(te_idx):
        w.writerow([rows[i]["message_id"], rows[i]["text_sha1"], int(y[i]), int(has_url[i])]
                   + [int(sum(p[j] for p in preds[k])) for k in arms])
check("8 arms", "fusion_test_predictions.csv byte-identical", same_file(VOTES_FILE, SAVED / "fusion_test_predictions.csv"), True)

## Bước 9: Khoảng tin cậy và bảng kết quả đầy đủ

Bản đăng ký ban đầu định dùng kiểm định Nadeau-Bengio trên 10 seed. Kiểm định này không dùng được ở đây. Nó được thiết kế cho trường hợp dữ liệu
được chia lại 10 lần, còn ở đây 10 seed chỉ đổi khởi tạo MLP trên cùng một tập test. Notebook vẫn tính nó (`corrected_paired_t`) vì kết quả đã
lưu giữ p-value cũ để lưu vết, đánh dấu `registered_inference_audit.valid = false`; bài không còn dùng các p-value này.

Thay vào đó, bài dùng bootstrap theo cụm nội dung tin (`cluster_bootstrap_f1`). Ý tưởng của bootstrap là giả lập "nếu lấy một tập test khác
cùng kiểu thì kết quả dao động bao nhiêu", bằng cách rút ngẫu nhiên có hoàn lại từ chính tập test hiện có. Cách làm cụ thể:
1. Mỗi nội dung tin khác nhau trong test là một cụm: 566 dòng test gom thành 512 cụm. Các bản sao của một tin luôn đi cùng cụm của nó, để một
   tin lặp lại không bị tính như nhiều mẫu độc lập.
2. Rút có hoàn lại 512 cụm, lặp 5.000 lần (seed 20260902).
3. Với mỗi lần rút, tính F1 của từng kênh trên các cụm được rút, lấy trung bình qua 10 seed, rồi lấy hiệu giữa hai kênh.
4. Sau 5.000 lần rút có 5.000 giá trị hiệu. Khoảng từ phân vị 2,5% đến 97,5% của chúng là CI 95%.

Để chạy nhanh, hàm không gọi `f1_score` 3 × 10 × 5.000 lần. Nó đếm TP/FP/FN của từng cụm một lần, rồi nhân với ma trận "mỗi lần rút, cụm nào
được rút bao nhiêu lần" để có TP/FP/FN của cả 5.000 lần rút cùng lúc.

Ô dưới dựng toàn bộ bảng kết quả của phép so sánh đã đăng ký (điểm từng kênh, độ lệch chuẩn, ma trận nhầm lẫn trung bình, FPR, AUC, kiểm định
cũ, bootstrap, kết quả theo nhóm) rồi so với `fusion_results.json`, từng trường.

In [ ]:
from scipy import stats

def corrected_paired_t(d, test_frac=0.30):
    # Corrected resampled t-test (kept only because the saved results record it)
    d = np.asarray(d, dtype=float)
    k = len(d); mean = float(np.mean(d)); var = float(np.var(d, ddof=1)) if k > 1 else 0.0
    out = {"k": k, "mean": mean, "wins": int(np.sum(d > 0)), "p": 1.0, "p_naive": 1.0, "se": 0.0, "inflation": float("nan")}
    if k < 2 or var <= 0:
        return out
    factor = 1.0 / k + test_frac / (1.0 - test_frac)
    se = math.sqrt(factor * var); se_naive = math.sqrt(var / k); t = mean / se
    out.update(p=float(2 * stats.t.sf(abs(t), k - 1)), p_naive=float(2 * stats.t.sf(abs(mean / se_naive), k - 1)),
               se=se, t=float(t), inflation=se / se_naive)
    return out

def cluster_bootstrap_f1(yy, pred_a, pred_b, grp, B=5000, seed=20260902):
    # Paired bootstrap over distinct test texts, duplicate rows kept inside their cluster
    yy = np.asarray(yy, dtype=np.int8)
    grp = np.asarray(grp)
    uniq = sorted(set(grp.tolist()))
    index = {g: i for i, g in enumerate(uniq)}
    gid = np.array([index[g] for g in grp])
    draws = np.random.default_rng(seed).multinomial(len(uniq), np.full(len(uniq), 1 / len(uniq)), size=B)
    def boot_f1(pp):
        scores_ = []
        for p in pp:
            p = np.asarray(p, dtype=np.int8)
            tp = np.bincount(gid, weights=((yy == 1) & (p == 1)), minlength=len(uniq))
            fp = np.bincount(gid, weights=((yy == 0) & (p == 1)), minlength=len(uniq))
            fn = np.bincount(gid, weights=((yy == 1) & (p == 0)), minlength=len(uniq))
            T_, Fp, Fn = draws @ tp, draws @ fp, draws @ fn
            scores_.append(np.divide(2 * T_, 2 * T_ + Fp + Fn, out=np.zeros(B), where=(2 * T_ + Fp + Fn) != 0))
        return np.mean(scores_, axis=0)
    diffs = boot_f1(pred_a) - boot_f1(pred_b)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    return {"mean": float(np.mean(diffs)), "ci95": [float(lo), float(hi)], "B": B, "clusters": len(uniq), "seed": seed,
            "unit": "distinct test text; duplicate rows retained within cluster",
            "status": "post-hoc sensitivity; conditional on the fitted seed ensemble"}

def contrast_set(pr, grp, yy):
    return {"T1_fusion_minus_url": cluster_bootstrap_f1(yy, pr["fusion"], pr["url"], grp),
            "T2_text_minus_url": cluster_bootstrap_f1(yy, pr["text"], pr["url"], grp),
            "posthoc_fusion_minus_text": cluster_bootstrap_f1(yy, pr["fusion"], pr["text"], grp)}

def results_table(res_, preds_, has_url_, rule, diag_):
    f1 = {k: np.array([m_["f1"] for m_ in v]) for k, v in res_.items()}
    keep_p = lambda t_: {kk: (vv if kk in ("p", "p_naive") else round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in t_.items()}
    return {
        "n_messages": len(rows), "n_train": int(tr_mask.sum()), "n_test": int(te_mask.sum()),
        "n_with_url": int(has_url_.sum()), "seeds": len(SEEDS), "url_rule": rule,
        "arms": {k: {m_: mean4(v, m_) for m_ in ("f1", "precision", "recall")} for k, v in res_.items()},
        "arms_sd": {k: round(float(np.std(f1[k], ddof=1)), 4) for k in f1},
        "confusion_mean": {k: {c: round(float(np.mean([x[c] for x in v])), 1) for c in ("tp", "fp", "fn", "tn")} for k, v in res_.items()},
        "fpr": {k: mean4(v, "fpr") for k, v in res_.items()},
        "auc": {k: {m_: mean4(v, m_) for m_ in ("roc_auc", "pr_auc")} for k, v in res_.items()},
        "T1_fusion_minus_url": keep_p(corrected_paired_t(f1["fusion"] - f1["url"])),
        "T2_text_minus_url": keep_p(corrected_paired_t(f1["text"] - f1["url"])),
        "posthoc_fusion_minus_text": keep_p(corrected_paired_t(f1["fusion"] - f1["text"])),
        "registered_inference_audit": {"valid": False,
            "reason": "Nadeau-Bengio requires repeated data splits; these ten values vary only the optimizer seed on one fixed holdout.",
            "legacy_seed_tests_retained_for_provenance": True},
        "cluster_bootstrap": contrast_set(preds_, groups, y[te_mask]),
        "diagnostics": diag_}

fusion = results_table(res, preds, has_url, "registered", diag)
for k, z in fusion["cluster_bootstrap"].items():
    print(f"{k:28} mean {z['mean']:+.4f}  CI95 [{z['ci95'][0]:+.4f}, {z['ci95'][1]:+.4f}]  ({z['clusters']} clusters)")
check_json("9 registered", "fusion_results.json", fusion, SAVED / "fusion_results.json")
report("8 arms")
report("9 registered")

CI của fusion trừ text chứa 0, nên kết luận "kênh URL gần như không đóng góp gì" vẫn đứng vững. CI của T1 và T2 thì nằm xa 0.

Giờ đã có embedding và kết quả theo nhóm, notebook chạy phần thí nghiệm theo lô của Bước 5 (với BLAS một luồng, khoảng 1 phút):

In [ ]:
with one_thread():
    batch = batch_audit(Xtxt, diag)
check_json("5 batch", "batch_audit.json", batch, SAVED / "batch_audit.json")

lb = batch["leave_batches_out"]
display(pd.DataFrame({"two batches held out": {**{f"probe {k}": v for k, v in lb["batch"]["probe"].items()},
                                               **{f"text arm {k}": v for k, v in lb["batch"]["text_arm"].items()}},
                      "random control [min, max]": {**{f"probe {k}": v for k, v in lb["control_span"]["probe"].items()},
                                                   **{f"text arm {k}": v for k, v in lb["control_span"]["text_arm"].items()}}}))
print("URL arm ceiling:", batch["url_ceiling"])
print("twins a text hash cannot see:", batch["twins"])
report("5 batch")

Đọc bảng của Bước 5: khi bỏ hai lô ra khỏi train, recall ở ngưỡng 0,5 của probe tụt còn khoảng 0,14, trong khi đối chứng giữ được khoảng 0,65 đến
0,69. Nhưng ROC-AUC vẫn khoảng 0,97. Nghĩa là mô hình vẫn xếp tin phishing cao hơn tin ham; chỉ có ngưỡng 0,5 không còn hợp vì tỷ lệ lớp lúc
train đã đổi. Đó là lý do có thêm recall ở FPR 2%.

`url_ceiling` là mức trần của kênh URL: 38,5% tin phishing trong test không có URL, nên kể cả khi kênh URL đoán đúng mọi tin có URL, recall cao
nhất cũng chỉ 0,615 và F1 cao nhất 0,762.

## Bước 10: Vì sao URL không giúp gì?

Giả thuyết ban đầu: URL thừa vì PhoBERT đã đọc URL nằm sẵn trong tin, nên 21 cột URL chỉ lặp lại thứ PhoBERT đã thấy. Cách thử: thay mọi URL
trong tin bằng `<URL>` (cùng regex `URL_RE`), tính lại embedding, rồi so ba trường hợp:
- text đầy đủ;
- text đã che URL. Nếu giả thuyết đúng, điểm phải tụt;
- text đã che URL, cộng thêm 21 cột URL. Nếu giả thuyết đúng, điểm phải hồi lại.

Kết quả: che URL gần như không đổi điểm (+0,006), thêm URL trở lại cũng gần như không đổi (+0,006), và cả hai CI đều chứa 0. Vậy giả thuyết sai.
Tín hiệu nằm ở phần chữ ngoài URL; còn URL, dù ở dạng chữ hay dạng đặc trưng, gần như không đóng góp gì.

In [ ]:
masked = [URL_RE.sub(" <URL> ", t) for t in texts]
print("messages changed by masking:", sum(a_ != b_ for a_, b_ in zip(texts, masked)))
Xmask = embed(masked, "masked")
res_m, preds_m = run_arms({"text_masked": Xmask, "masked+url": np.hstack([Xmask, Xurl])}, tr_mask, te_mask)
why = {"text_full": float(np.mean([x["f1"] for x in res["text"]])),
       "text_masked": float(np.mean([x["f1"] for x in res_m["text_masked"]])),
       "masked_plus_url": float(np.mean([x["f1"] for x in res_m["masked+url"]])),
       "mask_effect": cluster_bootstrap_f1(y[te_mask], preds_m["text_masked"], preds["text"], groups),
       "url_adds_to_masked": cluster_bootstrap_f1(y[te_mask], preds_m["masked+url"], preds_m["text_masked"], groups)}
print({k: round(v, 4) for k, v in why.items() if isinstance(v, float)})
for k in ("mask_effect", "url_adds_to_masked"):
    print(f"{k:20} {why[k]['mean']:+.4f}  CI95 {np.round(why[k]['ci95'], 4)}")
check_json("10 masking", "why_fusion.json", why, SAVED / "why_fusion.json")
report("10 masking")

## Bước 11: Luật trích URL chặt hơn

Luật trong bản đăng ký lấy chỗ khớp regex đầu tiên, nên 109 chỗ khớp thực ra là mẩu câu (`ngay.tcqc`, `tp.hcm`). Luật `psl` bỏ qua các mẩu
không kết thúc bằng đuôi tên miền thật và lấy URL hợp lệ đầu tiên. Kênh text giữ nguyên; chỉ kênh URL và fusion thay đổi. Kết quả được dựng
thành cùng một bảng như Bước 9 và so với `fusion_results_psl.json`.

In [ ]:
urls_psl = [first_url(t, "psl") for t in texts]
has_url_psl = np.array([bool(x) for x in urls_psl])
Xurl_psl = url_matrix(urls_psl)
print("messages whose first URL changes:", sum(a_ != b_ for a_, b_ in zip(urls, urls_psl)))
res_psl, preds_psl = run_arms({"url": Xurl_psl, "text": Xtxt, "fusion": np.hstack([Xtxt, Xurl_psl])}, tr_mask, te_mask)
diag_psl = {}
strata_psl = {"no_url": ~has_url_psl[te_mask], "has_url": has_url_psl[te_mask],
              "shortened_phish": np.array([rows[i]["message_id"] in short_ids for i in te_idx])}
for name in res_psl:
    for tag, mask in strata_psl.items():
        if mask.sum():
            diag_psl[f"{name}/{tag}"] = {
                "n": int(mask.sum()), "n_pos": int(y[te_mask][mask].sum()),
                "f1": round(float(np.mean([f1_score(y[te_mask][mask], p[mask], pos_label=1, zero_division=0) for p in preds_psl[name]])), 4),
                "detected_mean": round(float(np.mean([int(((y[te_mask][mask] == 1) & (p[mask] == 1)).sum()) for p in preds_psl[name]])), 2)}
fusion_psl = results_table(res_psl, preds_psl, has_url_psl, "psl", diag_psl)
display(pd.DataFrame({"registered rule": {k: v["f1"] for k, v in fusion["arms"].items()},
                      "psl rule": {k: v["f1"] for k, v in fusion_psl["arms"].items()}}))
check_json("11 psl", "fusion_results_psl.json", fusion_psl, SAVED / "fusion_results_psl.json")
report("11 psl")

## Bước 12: Chia lại để tin gần giống không nằm hai phía

Bước 4 cho thấy ở ngưỡng độ giống 0,90, vẫn có khoảng 34% tin ham và 19% tin phishing trong test có một tin rất giống nó trong train. Bước này
chia lại dữ liệu như sau:
1. Biểu diễn mỗi tin bằng TF-IDF trên các cụm 3 đến 5 ký tự, nối hai tin lại nếu độ giống cosine từ 0,90 trở lên, rồi gom thành nhóm. Hai tin
   cùng nhóm nếu có một chuỗi tin giống nhau nối chúng (thành phần liên thông).
2. Đưa nguyên cả nhóm vào train hoặc test, dựa trên hash của `message_id` nhỏ nhất trong nhóm. Đây là cùng ý tưởng với chia theo hash nội dung
   ở Bước 2, chỉ là áp cho nhóm thay vì cho từng tin. 25% số nhóm vào test.
3. Chạy lại ba kênh với 10 seed, bootstrap theo nhóm (thay vì theo nội dung tin), và tính lại sàn trên cách chia mới.

Ba kết cục được viết ra trước khi xem số: điểm gần như không đổi; điểm tụt nhưng thứ tự các kênh giữ nguyên; hoặc thứ tự đổi. Kết quả thực tế:
text vẫn hơn URL, và fusion vẫn xấp xỉ text.

In [ ]:
def split_by_component(comp, holdout):
    rep = {}
    for r, c in zip(rows, comp):
        rep[int(c)] = min(rep.get(int(c), r["message_id"]), r["message_id"])
    side = {}
    for c, key in rep.items():
        h = int(hashlib.sha1(("simsplit:" + key).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
        side[c] = "test" if h < holdout else "train"
    return np.array([side[int(c)] for c in comp])

comp = near_dup_components(texts, 0.90)
split_c = split_by_component(comp, 0.25)
trc, tec = split_c == "train", split_c == "test"
res_c, preds_c = run_arms(arms, trc, tec)

toks_c = sorted({t for r in rows for t in (r["pii_tokens"] or "").split()})
def shallow_c(r, with_all):
    v = [1.0 if t in set((r["pii_tokens"] or "").split()) else 0.0 for t in toks_c]
    return v + ([float(int(r["accented"] or 0)), int(r["chars"] or 0) / 100.0] if with_all else [])
floors = {}
for tag, with_all in (("tokens", False), ("all_shallow", True)):
    A = np.array([shallow_c(r, with_all) for r in rows], dtype=float)
    floors[tag] = round(float(f1_score(y[tec], LogisticRegression(max_iter=2000, random_state=0).fit(A[trc], y[trc]).predict(A[tec]))), 4)

robust = {"arms": {k: {m_: mean4(res_c[k], m_) for m_ in ("f1", "precision", "recall")} for k in arms},
          **contrast_set(preds_c, comp[tec], y[tec]),
          "split": {"sim": 0.9, "holdout": 0.25, "components": int(comp.max()) + 1, "n_train": int(trc.sum()), "n_test": int(tec.sum()),
                    "moved": int((split_c != split).sum()), "test_phishing": int(y[tec].sum()), "test_ham": int((1 - y[tec]).sum())},
          "floors": floors, "seeds": len(SEEDS), "registered": False,
          "note": "post-hoc sensitivity analysis; the registered split is the text-hash one"}
for k in arms:
    robust["arms"][k]["delta_vs_registered"] = round(robust["arms"][k]["f1"] - fusion["arms"][k]["f1"], 4)
robust["order_holds"] = bool(robust["arms"]["text"]["f1"] > robust["arms"]["url"]["f1"])
print(robust["split"], "| floors:", floors)
display(pd.DataFrame(robust["arms"]).T)
check_json("12 cluster split", "split_robustness.json", robust, SAVED / "split_robustness.json")
report("12 cluster split")

## Bước 13: Kiểm tra theo lô thời gian, và các chẩn đoán phụ

### 13a. Theo lô thời gian

Nhắc lại: `date` là ngày nhập lô. Các lô từ 01/08/2026 trở đi toàn là ham, nên phép thử chỉ đo được tỷ lệ báo nhầm (FPR) trên tin ham mới,
không đo được recall. Có ba lần chạy, đều dùng baseline TF-IDF của Bước 6:

| Lần chạy | Train | Test |
|---|---|---|
| `registered_august` | phần train của cách chia đã đăng ký | tin test thuộc lô tháng 8 |
| `without_august` | phần train, chỉ lấy lô trước tháng 8 | phần test, chỉ lấy lô trước tháng 8 (có cả phishing nên tính được F1) |
| `forward_august` | mọi tin trước tháng 8 | mọi tin từ tháng 8, bỏ các tin trùng nguyên văn với tin trước đó |

Kết quả kèm SHA-256 của file dữ liệu thô và của bảng tin, để biết chính xác nó được tính trên dữ liệu nào. Bảng tin ở đây là file notebook vừa
ghi ở Bước 2; vì file đó trùng từng byte với file đã lưu nên hai hash phải trùng.

In [ ]:
dates = [dt.datetime.strptime(r["date"], "%d/%m/%Y").date() if r["date"] else None for r in rows]
early = np.array([d is not None and d < dt.date(2026, 8, 1) for d in dates])
late = np.array([d is not None and d >= dt.date(2026, 8, 1) for d in dates])
assert np.all(y[late] == 0), "August-or-later rows were expected to be ham only"
overlap = late & np.isin(T_all, T_all[early])
masks = {"registered_august": (tr_mask, ~tr_mask & late), "without_august": (tr_mask & early, ~tr_mask & early),
         "forward_august": (early, late & ~overlap)}
temporal = {"status": "post-hoc retrospective sensitivity; one fitted seed", "cutoff": "2026-08-01",
            "undated_rows": sum(d is None for d in dates), "forward_excluded_exact_text_rows": int(overlap.sum()),
            "source_sha256": hashlib.sha256(SRC.read_bytes()).hexdigest(),
            "split_sha256": hashlib.sha256(MSG_FILE.read_bytes()).hexdigest(),
            "model": "word TF-IDF (1,2), min_df=2, sublinear_tf; LR C=4, seed=0, max_iter=3000", "runs": {}}
temporal_preds = []
for name, (a_m, b_m) in masks.items():
    assert not set(T_all[a_m]) & set(T_all[b_m]), name
    mdl = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                        LogisticRegression(C=4, max_iter=3000, random_state=0)).fit(T_all[a_m].tolist(), y[a_m])
    pr = (mdl.predict_proba(T_all[b_m].tolist())[:, 1] > .5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[b_m], pr, labels=[0, 1]).ravel()
    r_ = {"n_train": int(a_m.sum()), "n_test": int(b_m.sum()), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
          "fpr": float(fp / (tn + fp))}
    if np.any(y[b_m] == 1):
        r_.update(f1=float(f1_score(y[b_m], pr)), precision=float(precision_score(y[b_m], pr)), recall=float(recall_score(y[b_m], pr)))
    temporal["runs"][name] = r_
    for i, p in zip(np.flatnonzero(b_m), pr):
        temporal_preds.append({"run": name, "message_id": rows[i]["message_id"], "label": int(y[i]), "prediction": int(p)})
display(pd.DataFrame(temporal["runs"]).T)
check_json("13 temporal", "temporal_audit.json", temporal, SAVED / "temporal_audit.json")
with open(TMP / "temporal_predictions.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(temporal_preds[0])); w.writeheader(); w.writerows(temporal_preds)
check("13 temporal", "temporal_predictions.csv byte-identical", same_file(TMP / "temporal_predictions.csv", SAVED / "temporal_predictions.csv"), True)
report("13 temporal")

### 13b. URL dạng ký tự, cắt cụt token, phân tích lỗi

Bốn câu hỏi phụ:
- `url_char`: kênh URL yếu là do bản thân URL hay do 21 cột đặc trưng? Huấn luyện TF-IDF trên các cụm 2 đến 5 ký tự của chính chuỗi URL (theo
  luật psl), cộng LogReg. Trên tin có URL, mô hình này tốt hơn hẳn (F1 khoảng 0,87). Tính trên mọi tin test thì chỉ còn khoảng 0,66, vì tin không
  có URL bị đoán là ham. Vậy giới hạn chính là việc nhiều tin không có URL, không chỉ là bộ đặc trưng.
- `truncation`: thí nghiệm fine-tune cắt tin ở 128 token sau khi tách từ, còn kênh frozen cắt ở 256 token. Mỗi giới hạn cắt bao nhiêu phần trăm
  tin của mỗi lớp? Phần này cần tokenizer của PhoBERT.
- `fp_examples` và `fn_examples`: tin ham bị kênh text báo nhầm ở đa số seed, và tin phishing bị bỏ sót ở đa số seed, đọc từ bảng phiếu bầu ở
  Bước 8.
- `text_minus_floor`: kênh text trừ hai sàn gradient boosting của Bước 4, kèm CI bootstrap.

Kết quả đã lưu được tính với BLAS một luồng, nên phần này chạy trong `one_thread()`.

In [ ]:
from transformers import AutoTokenizer
from pyvi import ViTokenizer

def segment(text):
    # PhoBERT's expected input: word-segmented Vietnamese, syllables of a word joined by underscores
    return ViTokenizer.tokenize(text or "")

extra = {"registered": False, "status": "post-hoc, descriptive"}
with one_thread():
    U = np.array([first_url(t, "psl") for t in texts], dtype=object)
    has_p = np.array([bool(u_) for u_ in U])
    v = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    m_url = LogisticRegression(max_iter=3000, C=4, random_state=0).fit(v.fit_transform(U[tr_mask & has_p]), y[tr_mask & has_p])
    sel = te_mask & has_p
    prob = m_url.predict_proba(v.transform(U[sel]))[:, 1]
    p = (prob >= 0.5).astype(int)
    full = np.zeros(int(te_mask.sum()), dtype=int); full[has_p[te_mask]] = p
    extra["url_char"] = {"train_rows_with_url": int((tr_mask & has_p).sum()), "test_rows_with_url": int(sel.sum()),
                         "has_url_f1": round(float(f1_score(y[sel], p)), 4), "has_url_roc_auc": round(float(roc_auc_score(y[sel], prob)), 4),
                         "has_url_pr_auc": round(float(average_precision_score(y[sel], prob)), 4),
                         "all_test_f1": round(float(f1_score(y[te_mask], full)), 4),
                         "all_test_precision": round(float(precision_score(y[te_mask], full)), 4),
                         "all_test_recall": round(float(recall_score(y[te_mask], full)), 4),
                         "model": "char 2-5-gram TF-IDF on the first public-suffix URL, logistic regression C=4"}

    tokz, _ = phobert()
    raw_len = np.array([len(tokz(t)["input_ids"]) for t in texts])
    seg_len = np.array([len(tokz(segment(t))["input_ids"]) for t in texts])
    extra["truncation"] = {lab: {"raw_over_256_pct": round(100 * float((raw_len[y == k] > 256).mean()), 1),
                                 "seg_over_128_pct": round(100 * float((seg_len[y == k] > 128).mean()), 1),
                                 "seg_over_256_pct": round(100 * float((seg_len[y == k] > 256).mean()), 1),
                                 "seg_median_tokens": int(np.median(seg_len[y == k]))} for k, lab in ((0, "ham"), (1, "positive"))}

    pr_rows = list(csv.DictReader(open(VOTES_FILE, encoding="utf-8")))
    date_of = {r["message_id"]: r["date"] for r in rows}
    fps = sorted((r for r in pr_rows if r["label"] == "0" and int(r["text_votes"]) > 0), key=lambda r: -int(r["text_votes"]))
    seen, ex = set(), []
    for r in fps:
        if r["text_sha1"] in seen:
            continue
        seen.add(r["text_sha1"])
        ex.append({"message_id": r["message_id"], "text_votes": int(r["text_votes"]), "has_url": int(r["has_url"])})
    extra["fp_examples"] = {"ham_rows_flagged_by_any_seed": len(fps),
                            "ham_rows_flagged_by_majority": sum(1 for r in fps if int(r["text_votes"]) >= 6),
                            "distinct_texts_flagged_by_majority": sum(1 for e in ex if e["text_votes"] >= 6),
                            "majority_in_august_batch": sum(1 for r in fps if int(r["text_votes"]) >= 6
                                                            and date_of[r["message_id"]].endswith(("/8/2026", "/08/2026"))),
                            "top": ex[:8]}
    fns = [r for r in pr_rows if r["label"] == "1" and int(r["text_votes"]) <= 4]
    extra["fn_examples"] = {"positive_rows_missed_by_majority": len(fns),
                            "in_two_positive_batches": sum(1 for r in fns if date_of[r["message_id"]] in POS_BATCHES),
                            "without_url": sum(1 for r in fns if r["has_url"] == "0"), "ids": [r["message_id"] for r in fns][:12]}

    text_preds_1t = [fit_score(Xtxt[tr_mask], y[tr_mask], Xtxt[te_mask], y[te_mask], s_)[1] for s_ in SEEDS]
    fl = {r["message_id"]: r for r in csv.DictReader(open(FLOOR_FILE, encoding="utf-8"))}
    ids = [rows[i]["message_id"] for i in te_idx]
    extra["text_minus_floor"] = {k: cluster_bootstrap_f1(y[te_mask], text_preds_1t, [np.array([int(fl[m_][k]) for m_ in ids])], groups)
                                 for k in ("floor_full", "floor_strict")}

print("URL as characters:", {k: v for k, v in extra["url_char"].items() if k != "model"})
print("truncation:", extra["truncation"])
print("false positives:", {k: v for k, v in extra["fp_examples"].items() if k != "top"})
print("false negatives:", {k: v for k, v in extra["fn_examples"].items() if k != "ids"})
check_json("13 extra", "extra_diagnostics.json", extra, SAVED / "extra_diagnostics.json")
report("13 extra")

## Bước 14: Ba thí nghiệm "pre-decision" (16/09/2026, post-hoc, không đăng ký)

Đây là ba câu hỏi mà người phản biện dễ hỏi. Cả ba chạy trên đúng cách chia và các seed đã đăng ký.

### 14a. Bỏ hết dấu thì còn tách lớp được không?

`strip_diacritics` chuẩn hoá NFD, bỏ các dấu, rồi đổi `đ` thành `d`, nên "bạn" và "ban" trở thành cùng một từ. Huấn luyện lại các mô hình TF-IDF
(theo từ, theo ký tự, có và không có token che) và kênh PhoBERT trên văn bản đã bỏ dấu. Nếu điểm giữ gần như
cũ, tín hiệu nằm ở việc dùng từ nào, chứ không phải ở cách viết có dấu hay không. Chạy với BLAS một luồng như kết quả đã lưu.

In [ ]:
def strip_diacritics(text):
    t = unicodedata.normalize("NFD", text or "")
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    return t.replace("đ", "d").replace("Đ", "D")

def summarise(per_seed):
    keys = ("f1", "precision", "recall")
    return {k: round(float(np.mean([m_[k] for m_ in per_seed])), 4) for k in keys} | {
        "f1_sd": round(float(np.std([m_["f1"] for m_ in per_seed], ddof=1)), 4) if len(per_seed) > 1 else 0.0}

GRIDS = {"word": dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
         "char": dict(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)}

def linear(train_texts, ytr, test_texts, yte, grid, strip_tokens=False):
    prep = (lambda t: TOKRX.sub(" ", t)) if strip_tokens else (lambda t: t)
    v_ = TfidfVectorizer(**GRIDS[grid])
    A = v_.fit_transform([prep(t) for t in train_texts]); B_ = v_.transform([prep(t) for t in test_texts])
    p = LogisticRegression(max_iter=3000, C=4, random_state=0).fit(A, ytr).predict(B_)
    return round(float(f1_score(yte, p)), 4), p

for t in ("Quý khách đã đăng ký thành công", "Bạn được hoàn tiền"):
    print(f"{t!r:36} -> {strip_diacritics(t)!r}")

sel_ = lambda seq, m_: [x for x, k in zip(seq, m_) if k]
nodia = [strip_diacritics(t) for t in texts]
with one_thread():
    dia = {"n_changed_by_normalisation": sum(1 for t, n in zip(texts, nodia) if t != n), "linear": {}}
    lin_preds = {}
    for label, corpus in (("accented", texts), ("nodia", nodia)):
        for grid in ("word", "char"):
            f, p = linear(sel_(corpus, tr_mask), y[tr_mask], sel_(corpus, te_mask), y[te_mask], grid)
            dia["linear"][f"tfidf_{grid}_{label}"] = f; lin_preds[f"tfidf_{grid}_{label}"] = p
        f, p = linear(sel_(corpus, tr_mask), y[tr_mask], sel_(corpus, te_mask), y[te_mask], "word", strip_tokens=True)
        dia["linear"][f"tfidf_word_no_tokens_{label}"] = f; lin_preds[f"tfidf_word_no_tokens_{label}"] = p
    Xnd = embed(nodia, "nodia")
    res_d, preds_d = run_arms({"text": Xtxt, "text_nodia": Xnd}, tr_mask, te_mask)
dia["arms"] = {k: summarise(v) for k, v in res_d.items()}
dia["cluster_bootstrap"] = {
    "text_nodia_minus_text": cluster_bootstrap_f1(y[te_mask], preds_d["text_nodia"], preds_d["text"], groups),
    "tfidf_word_nodia_minus_accented": cluster_bootstrap_f1(y[te_mask], [lin_preds["tfidf_word_nodia"]], [lin_preds["tfidf_word_accented"]], groups),
    "tfidf_word_no_tokens_nodia_minus_accented": cluster_bootstrap_f1(
        y[te_mask], [lin_preds["tfidf_word_no_tokens_nodia"]], [lin_preds["tfidf_word_no_tokens_accented"]], groups)}
dia.update(seeds=len(SEEDS), normalisation="NFD, combining marks removed, đ/Đ -> d/D", status="post-hoc, unregistered")
display(pd.Series(dia["linear"]).to_frame("F1"))
print("PhoBERT arm:", {k: v["f1"] for k, v in dia["arms"].items()})
check_json("14 pre-decision", "diacritics_ablation.json", dia, SAVED / "diacritics_ablation.json")

### 14b. Fusion không giúp gì có phải do cách ghép?

Fusion đã đăng ký nối 21 cột URL vào 768 số của text rồi huấn luyện một MLP, trong khi 45% tin test không có URL (vector toàn 0). Có hai cách giải
thích: URL thật sự thừa, hoặc cách ghép này khiến MLP không tận dụng được URL. Thử thêm bốn cách ghép:
- `early_flag`: như fusion, thêm một cột cho biết tin có URL hay không;
- `late_mean`: huấn luyện riêng kênh URL và kênh text, rồi lấy trung bình hai xác suất;
- `late_gated`: tin không có URL thì chỉ dùng xác suất của text, tin có URL thì lấy trung bình;
- `stack`: huấn luyện thêm một LogReg nhận vào `(p_url, p_text, has_url)`. Để LogReg này không học trên xác suất "đã thấy đáp án", các xác suất
  dùng để huấn luyện nó được tính kiểu out-of-fold: chia train thành 5 phần, mỗi phần được chấm bởi mô hình huấn luyện trên 4 phần còn lại. Các
  phần được chia theo nội dung tin (`StratifiedGroupKFold`), để một tin lặp lại không nằm ở phần chấm chính nó.

Chạy với BLAS một luồng như kết quả đã lưu.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def mlp(seed):
    return MLPClassifier(hidden_layer_sizes=(128,), max_iter=400, random_state=seed, early_stopping=True, n_iter_no_change=15)

def fit_proba(Xtr, ytr, Xte, seed):
    sc = StandardScaler().fit(Xtr)
    return mlp(seed).fit(sc.transform(Xtr), ytr).predict_proba(sc.transform(Xte))[:, 1]

def oof_proba(X, yy, grp, seed):
    out = np.zeros(len(yy))
    for f, (i, j) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(X, yy, grp)):
        out[j] = fit_proba(X[i], yy[i], X[j], seed * 100 + f)
    return out

def metrics(yte, p):
    return {"f1": f1_score(yte, p, pos_label=1, zero_division=0), "precision": precision_score(yte, p, pos_label=1, zero_division=0),
            "recall": recall_score(yte, p, pos_label=1, zero_division=0)}

tr_groups = np.array([r["text_sha1"] for r in rows])[tr_mask]
Xflag = np.hstack([Xtxt, Xurl, has_url.astype(np.float32)[:, None]])
names_lf = ("text", "fusion", "early_flag", "late_mean", "late_gated", "stack")
per, preds_lf = {k: [] for k in names_lf}, {k: [] for k in names_lf}
with one_thread():
    for s_ in SEEDS:
        p_url = fit_proba(Xurl[tr_mask], y[tr_mask], Xurl[te_mask], s_)
        p_txt = fit_proba(Xtxt[tr_mask], y[tr_mask], Xtxt[te_mask], s_)
        _, p_fus = fit_score(Xfus[tr_mask], y[tr_mask], Xfus[te_mask], y[te_mask], s_)
        _, p_flag = fit_score(Xflag[tr_mask], y[tr_mask], Xflag[te_mask], y[te_mask], s_)
        mean_ = (p_url + p_txt) / 2
        gated = np.where(has_url[te_mask], mean_, p_txt)
        Ztr = np.column_stack([oof_proba(Xurl[tr_mask], y[tr_mask], tr_groups, s_), oof_proba(Xtxt[tr_mask], y[tr_mask], tr_groups, s_),
                               has_url[tr_mask]])
        Zte = np.column_stack([p_url, p_txt, has_url[te_mask]])
        out = {"text": (p_txt >= 0.5).astype(int), "fusion": p_fus, "early_flag": p_flag, "late_mean": (mean_ >= 0.5).astype(int),
               "late_gated": (gated >= 0.5).astype(int), "stack": LogisticRegression(max_iter=2000).fit(Ztr, y[tr_mask]).predict(Zte)}
        for k, p in out.items():
            per[k].append(metrics(y[te_mask], p)); preds_lf[k].append(p)
late_fusion = {"seeds": len(SEEDS), "n_test": int(te_mask.sum()), "n_test_no_url": int((~has_url[te_mask]).sum()),
               "arms": {k: summarise(v) for k, v in per.items()},
               "cluster_bootstrap": {f"{k}_minus_text": cluster_bootstrap_f1(y[te_mask], preds_lf[k], preds_lf["text"], groups)
                                     for k in names_lf if k != "text"},
               "has_url_only": {k: round(float(np.mean([f1_score(y[te_mask][has_url[te_mask]], p[has_url[te_mask]], zero_division=0)
                                                        for p in preds_lf[k]])), 4) for k in names_lf},
               "stack_folds": "StratifiedGroupKFold(5) by message text",
               "status": "post-hoc, unregistered; the registered fusion is the concatenation MLP"}
display(pd.DataFrame({"F1": {k: v["f1"] for k, v in late_fusion["arms"].items()},
                      "minus text (bootstrap mean)": {k.replace("_minus_text", ""): round(z["mean"], 4)
                                                      for k, z in late_fusion["cluster_bootstrap"].items()}}))
check_json("14 pre-decision", "late_fusion.json", late_fusion, SAVED / "late_fusion.json")

### 14c. Dùng PhoBERT đúng cách

PhoBERT được huấn luyện trên tiếng Việt đã tách từ (ví dụ "Quý_khách"), nhưng kênh text đã đăng ký đưa vào văn bản chưa tách từ. Thí nghiệm này
có hai phần:
- `text_seg`: PhoBERT frozen trên văn bản đã tách từ bằng `pyvi`, mọi thứ khác giữ nguyên;
- `text_ft`: huấn luyện lại toàn bộ PhoBERT (fine-tune): 3 epoch, lr 2e-5, batch 8, cắt ở 128 token, 5 seed.

Fine-tune mất khoảng 20 phút mỗi seed, và dự đoán của nó không được lưu lại. Vì vậy notebook không làm lại `text_ft` và hai bootstrap có chữ
`ft`; các trường đó được bỏ khỏi phép so. Phần còn lại được tính lại ở ô dưới: kênh text và `text_seg` với 5 seed, TF-IDF, và hai bootstrap
không dùng `text_ft`.

In [ ]:
ft_saved = saved("phobert_ft.json")
n_ft = ft_saved["seeds"]
print("pyvi segmentation:", segment("Quý khách vui lòng đăng ký thông tin"))
Xseg = embed([segment(t) for t in texts], "seg")
res_s, preds_s = run_arms({"text": Xtxt, "text_seg": Xseg}, tr_mask, te_mask, seeds=SEEDS[:n_ft])
p_tfidf = probe().fit(Xtr_t, y[tr_mask]).predict(Xte_t)
ft_mine = {"seeds": n_ft, "n_train": int(tr_mask.sum()), "n_test": int(te_mask.sum()),
           "arms": {k: summarise(v) for k, v in res_s.items()},
           "tfidf_word_f1": round(float(f1_score(y[te_mask], p_tfidf)), 4),
           "cluster_bootstrap": {"seg_minus_text": cluster_bootstrap_f1(y[te_mask], preds_s["text_seg"], preds_s["text"], groups),
                                 "tfidf_minus_text": cluster_bootstrap_f1(y[te_mask], [p_tfidf], preds_s["text"], groups)}}
# Compare only the fields recomputed here: the fine-tuned arm and its two contrasts are left out
ref = json.loads(json.dumps(ft_saved))
ref["arms"].pop("text_ft"); ref["cluster_bootstrap"].pop("ft_minus_text"); ref["cluster_bootstrap"].pop("ft_minus_tfidf")
for k in ("finetune", "status"):
    ref.pop(k)
(TMP / "phobert_ft_without_ft.json").write_text(json.dumps(ref))
check_json("14 pre-decision", "phobert_ft.json without the fine-tuned arm", ft_mine, TMP / "phobert_ft_without_ft.json")
display(pd.DataFrame(ft_saved["arms"]).T)
report("14 pre-decision")

## Bước 15: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước. Cột `match` là `True` khi kết quả notebook tính ra bằng kết quả đã lưu. Phần lớn phép kiểm so nguyên một
file kết quả, từng trường một: `sms_snapshot.json`, `shallow_cues.json`, `batch_audit.json`, `fusion_results.json`, `why_fusion.json`,
`fusion_results_psl.json`, `split_robustness.json`, `temporal_audit.json`, `extra_diagnostics.json`, `diacritics_ablation.json`,
`late_fusion.json` và phần không fine-tune của `phobert_ft.json`. Mỗi dòng như vậy phủ từ vài chục tới vài trăm trường, và khớp có nghĩa là
không trường nào lệch. Thêm vào đó là năm file so từng byte (hai bảng tin và URL, dự đoán của sàn, bảng phiếu bầu, dự đoán theo lô thời gian).

Notebook không làm lại:
- thí nghiệm PhoBERT fine-tune (`text_ft`) và hai bootstrap `ft_minus_text`, `ft_minus_tfidf` (Bước 14c);
- các hình và bảng của bài;
- `size_mb` và thời gian chạy trong `text_linear.json`, vì chúng phụ thuộc máy.

Giới hạn tái lập: khớp tuyệt đối chỉ được đảm bảo khi chạy trên cùng máy, cùng phiên bản thư viện và cùng chế độ luồng BLAS như lúc sinh kết quả
đã lưu. Trên máy khác, các kênh MLP có thể lệch ở chữ số thứ ba.

In [ ]:
tbl = pd.DataFrame(CHECKS)
per_step = tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum")
per_step["mismatched"] = per_step["checks"] - per_step["matched"]
display(per_step)
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")
bad = tbl[~tbl["match"]]
if len(bad):
    display(bad)

In [ ]:
# Every individual check, for reference
with pd.option_context("display.max_rows", 500):
    display(tbl)